# 온라인 PINN PDE 프레임워크 V4

저장 결과 조회와 독립 순·역문제 비교를 제공합니다. 정의 코드는 접힌 셀에 있으며 마지막 패널에서 기능을 선택합니다.

`V4_Results.zip`은 V3 과거 결과와 V4 기능·검사 자료를 포함합니다. V4 본 실험은 새 학습으로 생성합니다.

## 코드 정의

In [ ]:
import sys
import tempfile
from pathlib import Path

V4_CODE_ROOT = Path(tempfile.mkdtemp(prefix="pinn_v4_code_"))
V4_PACKAGE = V4_CODE_ROOT / "v4"
V4_PACKAGE.mkdir()
V4_MODULE_DIR = V4_PACKAGE.as_posix()
sys.path.insert(0, str(V4_CODE_ROOT))
for module_name in tuple(sys.modules):
    if module_name == "v4" or module_name.startswith("v4."):
        del sys.modules[module_name]


In [ ]:
%%writefile "{V4_MODULE_DIR}/__init__.py"
"""V4 experiment components. Importing this package does not start training."""

from .generalization import GeneralizationPlan, field_metrics
from .config import Settings

__all__ = ["GeneralizationPlan", "field_metrics", "Settings"]


In [ ]:
%%writefile "{V4_MODULE_DIR}/api_routes.py"
"""HTTP adapters. Training is not exposed; mutation routes default to disabled."""

from fastapi import FastAPI, HTTPException
from .api_schemas import Prediction, Question, Registration, Tool
from .reporting import comparison


def build_app(service, allow_registry_write=False):
    app = FastAPI(title="Online PINN PDE Framework V4")

    def safe(call):
        try:
            return call()
        except (ValueError, PermissionError, TypeError) as exc:
            raise HTTPException(status_code=400, detail=str(exc)) from None

    app.get("/health")(
        lambda: {"status": "ok", "version": "v4", "training_endpoint": False}
    )
    app.get("/models")(service.models)
    app.get("/comparison")(lambda: comparison(service.root))
    app.get("/model/info")(lambda trial_id: safe(lambda: service.model_info(trial_id)))

    @app.post("/predict")
    @app.post("/predict/batch")
    def predict(request: Prediction):
        return safe(lambda: service.predict(request.trial_id, request.coordinates))

    @app.post("/agent/ask")
    def ask(request: Question):
        return safe(lambda: service.ask(**request.model_dump()))

    @app.post("/agent/tool")
    def tool(request: Tool):
        if (
            request.name in {"generate_report", "generate_feature_catalog"}
            and not allow_registry_write
        ):
            raise HTTPException(403, "Document writes disabled")
        return safe(lambda: service.tool(**request.model_dump()))

    @app.get("/rag/search")
    @app.get("/rag/context")
    def search(query: str, dense: bool = False, backend: str = "numpy"):
        return safe(lambda: service.search(query, dense, backend))

    @app.get("/report")
    def report():
        p = service.root / "REPORT.md"
        return {"text": p.read_text(encoding="utf-8") if p.exists() else ""}

    @app.post("/registry/select")
    def select(request: Registration):
        if not allow_registry_write:
            raise HTTPException(403, "Registry writes disabled")
        return safe(
            lambda: service.register(
                request.equation, request.trial_id, request.approved
            )
        )

    @app.post("/registry/rollback")
    def rollback(equation: str, approved: bool = False):
        if not allow_registry_write:
            raise HTTPException(403, "Registry writes disabled")
        return safe(lambda: service.rollback(equation, approved))

    @app.post("/predict/active")
    def active(equation: str, coordinates: list[list[float]]):
        trial = service.registry()["active"].get(equation)
        if trial is None:
            raise HTTPException(404, "No active model")
        return safe(lambda: service.predict(trial, coordinates))

    return app


In [ ]:
%%writefile "{V4_MODULE_DIR}/api_schemas.py"
"""Public API request contracts; imported only when serving or testing the API."""

from typing import Literal
from pydantic import BaseModel, Field, ConfigDict


class Request(BaseModel):
    model_config = ConfigDict(extra="forbid")


class Prediction(Request):
    trial_id: str = Field(pattern="^[0-9a-f]{16}$")
    coordinates: list[list[float]] = Field(min_length=1, max_length=1024)


class Question(Request):
    question: str = Field(min_length=1, max_length=2000)
    use_llm: bool = False
    dense: bool = False
    backend: Literal["numpy", "faiss"] = "numpy"


class Registration(Request):
    equation: str
    trial_id: str = Field(pattern="^[0-9a-f]{16}$")
    approved: bool = False


class Tool(Request):
    name: Literal[
        "comparison",
        "models",
        "model_info",
        "predict",
        "search",
        "generate_report",
        "generate_feature_catalog",
    ]
    arguments: dict = Field(default_factory=dict)
    approved: bool = False


In [ ]:
%%writefile "{V4_MODULE_DIR}/balance.py"
"""Independent surface/volume quadrature for integral PDE balance diagnostics."""

import numpy as np
from .equations import EQUATIONS
from .storage import arrays_sha


def balance_report(runtime, resolution=8):
    name = runtime.settings.equation
    spec = EQUATIONS[name]
    bounds = np.asarray(runtime.source.domain, float)
    time_axis = spec["time_axis"]
    spatial = [a for a in range(len(bounds)) if a != time_axis]
    widths = np.ptp(bounds, axis=1)
    volume = float(np.prod(widths[spatial]))
    coefficient = float(runtime.coefficient)
    slices = (
        [None]
        if time_axis is None
        else bounds[time_axis, 0]
        + widths[time_axis]
        * np.array(
            [0.85, 0.925, 1.0]
            if runtime.settings.generalization == "time_extrapolation"
            else [0.2, 0.5, 0.8]
        )
    )
    axes = [
        bounds[a, 0] + widths[a] * (np.arange(resolution) + 0.5) / resolution
        for a in spatial
    ]
    space = np.stack(np.meshgrid(*axes, indexing="ij"), axis=-1).reshape(
        -1, len(spatial)
    )

    def points(values, time):
        x = np.zeros((len(values), len(bounds)))
        x[:, spatial] = values
        if time_axis is not None:
            x[:, time_axis] = time
        return x

    def terms(x):
        q, g, h = runtime.derivatives(x)
        q, g, h = q.numpy(), g.numpy(), h.numpy()
        f = runtime.source.forcing(x)
        if len(spec["outputs"]) == 3:
            flux = np.zeros((len(x), 3, 2))
            velocity = q[:, :2]
            for i in range(2):
                for j in range(2):
                    flux[:, i, j] = (
                        q[:, i] * velocity[:, j]
                        + (q[:, 2] if i == j else 0)
                        - coefficient * g[:, i, j]
                    )
            flux[:, 2, :] = velocity
            div = g[:, 0, 0] + g[:, 1, 1]
            bulk = -f.copy()
            for i in range(2):
                bulk[:, i] += (g[:, i, time_axis] if time_axis is not None else 0) - q[
                    :, i
                ] * div
            return flux, bulk
        if name == "elasticity2d":
            lam = coefficient * 0.25 / (1.25 * 0.5)
            mu = coefficient / 2.5
            trace = g[:, 0, 0] + g[:, 1, 1]
            stress = np.stack(
                [
                    np.stack(
                        [
                            lam * trace + 2 * mu * g[:, 0, 0],
                            mu * (g[:, 0, 1] + g[:, 1, 0]),
                        ],
                        axis=1,
                    ),
                    np.stack(
                        [
                            mu * (g[:, 0, 1] + g[:, 1, 0]),
                            lam * trace + 2 * mu * g[:, 1, 1],
                        ],
                        axis=1,
                    ),
                ],
                axis=1,
            )
            return -stress, -f
        flux = -coefficient * g[:, 0, spatial]
        bulk = -f[:, 0]
        if name == "poisson2d":
            flux = -g[:, 0, spatial]
            bulk = -coefficient * f[:, 0]
        elif name == "helmholtz":
            flux = -g[:, 0, spatial]
            bulk -= coefficient * q[:, 0]
        elif name == "darcy2d":
            flux *= (1 + 0.5 * np.sin(np.pi * x[:, 0]) * np.sin(np.pi * x[:, 1]))[
                :, None
            ]
        elif name == "burgers":
            flux[:, 0] += q[:, 0] ** 2 / 2
            bulk += g[:, 0, time_axis]
        elif name == "wave2d":
            bulk += h[:, 0, time_axis, time_axis]
        else:
            bulk += g[:, 0, time_axis]
            if name == "reaction_diffusion":
                bulk += -q[:, 0] + q[:, 0] ** 3
            if name == "advection_diffusion":
                flux += q[:, 0:1] * np.array([0.5, 0.25])
        return flux[:, None, :], bulk[:, None]

    before = runtime.snapshot()
    results = []
    all_coordinates = []
    for time in slices:
        x = points(space, time)
        _, bulk = terms(x)
        all_coordinates.append(x)
        integrated_bulk = volume * bulk.mean(axis=0)
        surface = np.zeros_like(integrated_bulk)
        for j, a in enumerate(spatial):
            other = [i for i in range(len(spatial)) if i != j]
            face = (
                space[::resolution]
                if len(spatial) == 2 and j == 1
                else space[:resolution] if len(spatial) == 2 else space[:1]
            )
            for end, sign in ((0, -1), (1, 1)):
                face = face.copy()
                face[:, j] = bounds[a, end]
                if other:
                    face[:, other[0]] = axes[other[0]]
                fx = points(face, time)
                flux, _ = terms(fx)
                all_coordinates.append(fx)
                surface += sign * volume / widths[a] * flux[:, :, j].mean(axis=0)
        defect = integrated_bulk + surface
        results.append(
            {
                "time": None if time is None else float(time),
                "bulk_integral": integrated_bulk.tolist(),
                "outgoing_flux_integral": surface.tolist(),
                "balance_defect": defect.tolist(),
                "normalized_balance_defect": (
                    defect / (np.asarray(runtime.scales["residual"]) * volume)
                ).tolist(),
            }
        )
    runtime.assert_unchanged(before)
    return {
        "definition": "volume source/time terms + separately sampled outgoing face flux; midpoint quadrature",
        "spatial_region": "full rectangular spatial domain; interpolation field holdout is reported separately",
        "resolution_per_axis": resolution,
        "coordinates_sha": arrays_sha({"points": np.concatenate(all_coordinates)}),
        "slices": results,
        "used_for_training_or_stopping": False,
        "state_guard": "PASS",
    }


In [ ]:
%%writefile "{V4_MODULE_DIR}/config.py"
"""Validated V4 settings and independent experiment identities."""

from dataclasses import asdict, dataclass, replace
import hashlib
import json
import math

ARMS = ("baseline", "loss_sa", "ff", "ff_loss_sa", "ff_curriculum", "supervised_only")
ARM_LABELS = {
    "baseline": "기본 PINN",
    "loss_sa": "손실항 SA",
    "ff": "푸리에 특징",
    "ff_loss_sa": "푸리에 특징 + 손실항 SA",
    "ff_curriculum": "푸리에 특징 + 커리큘럼",
    "supervised_only": "관측값 회귀",
}
SEEDS = (3234, 3235, 3236, 3237)


def digest(value):
    return hashlib.sha256(
        json.dumps(
            value, sort_keys=True, separators=(",", ":"), allow_nan=False
        ).encode()
    ).hexdigest()


@dataclass(frozen=True)
class Settings:
    equation: str = "wave2d"
    inverse: bool = False
    arms: tuple = ("baseline",)
    seeds: tuple = (3234,)
    cap: int = 100000
    target: float = 0.05
    stop_first: int = 15000
    check_every: int = 1000
    consecutive: int = 3
    sampling_every: int = 5000
    diagnostic_every: int = 1000
    base_lr: float = 0.004
    initial_coefficient: float | None = None
    width: int = 64
    depth: int = 5
    ff_features: int = 32
    ff_scales: tuple = (0.125, 0.25, 0.5)
    pde_count: int = 800
    observation_count: int = 200
    boundary_count: int = 80
    initial_count: int = 160
    validation_count: int = 256
    evaluation_count: int = 1024
    rad_candidates: int = 4000
    rad_power: float = 1.0
    rad_offset: float = 1.0
    sa_every: int = 50
    sa_ema: float = 0.9
    sa_bounds: tuple = (0.25, 4.0)
    lbfgs_maxiter: int = 500
    generalization: str = "basic"
    block_override: int | None = None
    data_mode: str = "synthetic"
    noise_fraction: float = 0.0
    revision: str = "v4-independent-1"

    def validate(self):
        from .equations import EQUATIONS

        if self.equation not in EQUATIONS or self.generalization not in {
            "basic",
            "interpolation",
            "time_extrapolation",
        }:
            raise ValueError("방정식 또는 일반화 모드를 확인하세요.")
        if type(self.inverse) is not bool or self.data_mode not in {"synthetic", "csv"}:
            raise ValueError("순·역문제 및 데이터 모드를 확인하세요.")
        if (
            not self.arms
            or len(set(self.arms)) != len(self.arms)
            or any(a not in ARMS for a in self.arms)
        ):
            raise ValueError("비교 방법이 올바르지 않습니다.")
        if (
            not self.seeds
            or len(set(self.seeds)) != len(self.seeds)
            or any(type(s) is not int or s < 0 for s in self.seeds)
        ):
            raise ValueError("시드는 서로 다른 음이 아닌 정수여야 합니다.")
        integers = [
            "cap",
            "stop_first",
            "check_every",
            "consecutive",
            "sampling_every",
            "diagnostic_every",
            "width",
            "depth",
            "ff_features",
            "pde_count",
            "observation_count",
            "boundary_count",
            "initial_count",
            "validation_count",
            "evaluation_count",
            "rad_candidates",
            "sa_every",
        ]
        if any(
            type(getattr(self, n)) is not int or getattr(self, n) <= 0 for n in integers
        ):
            raise ValueError("횟수·크기 설정은 양의 정수여야 합니다.")
        if (
            self.cap > 1000000
            or self.rad_candidates < self.pde_count
            or self.rad_candidates > 1000000
        ):
            raise ValueError("학습 한도 또는 RAD 후보 수가 지원 범위를 벗어납니다.")
        if type(self.lbfgs_maxiter) is not int or not 0 <= self.lbfgs_maxiter <= 10000:
            raise ValueError("L-BFGS 반복 수는 0~10000입니다.")
        for n in ("base_lr", "target", "rad_power", "rad_offset"):
            if not math.isfinite(getattr(self, n)) or getattr(self, n) <= 0:
                raise ValueError(n + " must be finite and positive")
        if (
            not 0 < self.target < 1
            or not 0 <= self.sa_ema < 1
            or not 0 <= self.noise_fraction < 1
        ):
            raise ValueError("오차 목표·EMA·노이즈 범위를 확인하세요.")
        if (
            len(self.sa_bounds) != 2
            or not 0 < self.sa_bounds[0] <= self.sa_bounds[1]
            or not all(math.isfinite(v) for v in self.sa_bounds)
        ):
            raise ValueError("SA 범위를 확인하세요.")
        if not self.ff_scales or any(
            not math.isfinite(v) or v <= 0 for v in self.ff_scales
        ):
            raise ValueError("FF scale을 확인하세요.")
        if self.initial_coefficient is not None and (
            not math.isfinite(self.initial_coefficient) or self.initial_coefficient <= 0
        ):
            raise ValueError("초기 계수는 양수여야 합니다.")
        if (
            self.generalization == "time_extrapolation"
            and EQUATIONS[self.equation]["time_axis"] is None
        ):
            raise ValueError("정상 문제에는 시간 외삽을 사용할 수 없습니다.")
        if self.block_override is not None and (
            type(self.block_override) is not int or not 0 <= self.block_override < 16
        ):
            raise ValueError("구역 번호는 0~15입니다.")
        if self.equation == "cylinder_ns" and self.data_mode != "csv":
            raise ValueError("원통 후류에는 외부 데이터 연결이 필요합니다.")
        return self

    def to_dict(self):
        return asdict(self)

    @classmethod
    def from_dict(cls, value):
        converted = dict(value)
        for name in ("arms", "seeds", "ff_scales", "sa_bounds"):
            if name in converted:
                converted[name] = tuple(converted[name])
        return cls(**converted).validate()

    def smoke(self):
        return replace(
            self,
            cap=10,
            stop_first=15,
            check_every=5,
            diagnostic_every=5,
            sampling_every=5,
            width=12,
            depth=2,
            ff_features=4,
            pde_count=16,
            observation_count=24,
            boundary_count=8,
            initial_count=12,
            validation_count=32,
            evaluation_count=64,
            rad_candidates=64,
            sa_every=2,
            lbfgs_maxiter=2,
        ).validate()


def learning_rates(completed, base):
    import numpy as np

    return tuple(
        base * float(np.interp(completed, [0, 5000, 10000], ratios))
        for ratios in ([1, 0.75, 0.5], [0.001, 0.75, 1])
    )


def group_weights(arm, completed, factors):
    import numpy as np

    if arm == "supervised_only":
        return np.array([0.0, 1.0, 0.0], np.float32)
    weights = np.ones(3, np.float64)
    if arm in {"loss_sa", "ff_loss_sa"}:
        weights *= factors
    if arm == "ff_curriculum":
        share = min(completed // 100 / 100, 1.0) / 3
        return np.array([share, (1 - share) / 2, (1 - share) / 2], np.float32)
    return (weights / weights.sum()).astype(np.float32)


def checkpoint_due(iteration, cap):
    return iteration in {max(1, math.ceil(cap * i / 5)) for i in range(1, 6)}


def target_reached(history, settings):
    eligible = [
        r
        for r in history
        if r["iteration"] >= settings.stop_first
        and r["iteration"] % settings.check_every == 0
    ]
    tail = eligible[-settings.consecutive :]
    return len(tail) == settings.consecutive and all(
        r["validation"]["relative_l1"] is not None
        and r["validation"]["relative_l2"] is not None
        and max(r["validation"]["relative_l1"], r["validation"]["relative_l2"])
        <= settings.target
        for r in tail
    )


In [ ]:
%%writefile "{V4_MODULE_DIR}/data.py"
"""Synthetic data or explicit CSV contracts; no hidden synthetic fill in CSV mode."""

import csv
import json
from pathlib import Path
import numpy as np

from .equations import EQUATIONS, exact_field, known_forcing
from .generalization import GeneralizationPlan
from .sampling import lhs, permitted_lhs, rng_for
from .storage import sha_file


def make_plan(settings, seed, domain=None):
    spec = EQUATIONS[settings.equation]
    return GeneralizationPlan(
        domain or spec["domain"],
        tuple(a for a in range(len(spec["axes"])) if a != spec["time_axis"]),
        spec["time_axis"],
        settings.generalization,
        seed,
        settings.block_override,
    )


def read_csv(path, axes, outputs, columns=None, transforms=None):
    columns = columns or {}
    transforms = transforms or {}
    with Path(path).open(encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream)
        header = reader.fieldnames or []
        rows = list(reader)
    if not rows:
        raise ValueError("관측 CSV가 비었습니다.")

    def column(name, optional=False):
        key = columns.get(name, name)
        if key not in header:
            if optional:
                return np.full(len(rows), np.nan)
            raise ValueError("CSV 열 없음: " + key)
        try:
            values = np.array(
                [float(r[key]) if r[key].strip() else np.nan for r in rows]
            )
        except (ValueError, KeyError):
            raise ValueError("CSV 숫자 형식 오류: " + key) from None
        scale, offset = transforms.get(name, [1.0, 0.0])
        if not np.isfinite([scale, offset]).all() or scale == 0:
            raise ValueError("단위 변환 오류")
        return values * scale + offset

    x = np.column_stack([column(a) for a in axes])
    y = np.column_stack([column(a, True) for a in outputs])
    if (
        not np.isfinite(x).all()
        or np.isinf(y).any()
        or not np.isfinite(y).any(axis=1).all()
    ):
        raise ValueError("CSV 좌표·관측값이 유효하지 않습니다.")
    if len({tuple(v) for v in x}) != len(x):
        raise ValueError("CSV에 중복 좌표가 있습니다. 집계 후 연결하세요.")
    return x, y


class DataSource:
    def __init__(self, settings, contract_path=None):
        self.settings = settings
        self.spec = EQUATIONS[settings.equation]
        self.external = {}
        self.contract = {}
        self.file_hashes = {}
        if settings.data_mode == "csv":
            if not contract_path:
                raise ValueError("데이터 연결 JSON이 필요합니다.")
            path = Path(contract_path)
            self.contract = json.loads(path.read_text(encoding="utf-8-sig"))
            c = self.contract
            if (
                c.get("equation") != settings.equation
                or c.get("canonical_units_confirmed") is not True
            ):
                raise ValueError("방정식·단위 확인이 필요합니다.")
            if (
                "domain" not in c
                or "fixed_coefficient" not in c
                or "field_scales" not in c
                or "residual_scales" not in c
            ):
                raise ValueError("영역·알려진 계수·정규화 척도를 지정하세요.")
            for family, name in c.get("files", {}).items():
                if family not in {
                    "observations",
                    "validation",
                    "evaluation",
                    "boundary",
                    "initial",
                    "initial_velocity",
                    "gauge",
                    "forcing",
                }:
                    raise ValueError("지원하지 않는 데이터 가족")
                file = (path.parent / name).resolve()
                if not file.is_relative_to(path.parent.resolve()):
                    raise ValueError("데이터 CSV는 계약 JSON의 폴더 안에 두세요.")
                self.file_hashes[family] = sha_file(file)
                outputs = (
                    self.residual_names
                    if family == "forcing"
                    else (
                        ["u_t"]
                        if family == "initial_velocity"
                        else self.spec["outputs"]
                    )
                )
                self.external[family] = read_csv(
                    file,
                    self.spec["axes"],
                    outputs,
                    c.get("columns", {}).get(family),
                    c.get("transforms", {}).get(family),
                )
            if not {"observations", "validation"} <= self.external.keys():
                raise ValueError("학습 관측과 별도 검증 CSV가 필요합니다.")
            if (
                set(settings.arms) != {"supervised_only"}
                and "boundary" not in self.external
            ):
                raise ValueError("PINN에는 경계조건 CSV가 필요합니다.")
            if (
                self.spec["time_axis"] is not None
                and set(settings.arms) != {"supervised_only"}
                and "initial" not in self.external
            ):
                raise ValueError("시간 문제에는 초기조건 CSV가 필요합니다.")
            if (
                settings.equation == "wave2d"
                and set(settings.arms) != {"supervised_only"}
                and "initial_velocity" not in self.external
            ):
                raise ValueError("파동 문제에는 초기 속도 u_t CSV가 필요합니다.")
            if len(self.spec["outputs"]) == 3 and set(settings.arms) != {
                "supervised_only"
            }:
                if (
                    "gauge" not in self.external
                    or not np.isfinite(self.external["gauge"][1][:, 2]).all()
                ):
                    raise ValueError(
                        "Navier–Stokes에는 알려진 압력 gauge CSV가 필요합니다."
                    )
            if c.get("forcing_mode") not in {"zero", "csv"}:
                raise ValueError("알려진 외력 zero/csv를 지정하세요.")
            if c["forcing_mode"] == "csv" and "forcing" not in self.external:
                raise ValueError("외력 CSV가 필요합니다.")
            seen = set()
            for family in ("observations", "validation", "evaluation"):
                if family in self.external:
                    points = self.external[family][0]
                    ids = {tuple(row) for row in points}
                    if ids & seen:
                        raise ValueError("관측·검증·평가 CSV 사이 좌표 누수")
                    seen |= ids
            for name, length in [
                ("field_scales", len(self.spec["outputs"])),
                ("residual_scales", len(self.residual_names)),
            ]:
                a = np.array(c[name], float)
                if a.shape != (length,) or not np.isfinite(a).all() or np.any(a <= 0):
                    raise ValueError(name + " 크기·양수 조건 오류")
            if not np.isfinite(c["fixed_coefficient"]) or c["fixed_coefficient"] <= 0:
                raise ValueError("알려진 계수는 양수여야 합니다.")
            if c.get("reference_coefficient") is not None and (
                not np.isfinite(c["reference_coefficient"])
                or c["reference_coefficient"] <= 0
            ):
                raise ValueError("평가용 참조 계수는 양수여야 합니다.")
        self.domain = self.contract.get("domain", self.spec["domain"])
        plan = make_plan(settings, settings.seeds[0], self.domain)
        for family, (x, y) in self.external.items():
            plan.coordinates(x)
            if family in {"initial", "initial_velocity"}:
                axis = self.spec["time_axis"]
                if axis is None or not np.allclose(
                    x[:, axis], self.domain[axis][0], rtol=0, atol=1e-10
                ):
                    raise ValueError("초기조건 좌표는 영역 시작 시간이어야 합니다.")
            if family == "boundary" and settings.equation != "cylinder_ns":
                domain = np.array(self.domain)
                spatial = list(plan.spatial_axes)
                on_edge = np.isclose(
                    x[:, spatial], domain[spatial, 0], atol=1e-10, rtol=0
                ) | np.isclose(x[:, spatial], domain[spatial, 1], atol=1e-10, rtol=0)
                if not on_edge.any(axis=1).all():
                    raise ValueError("경계조건 좌표가 공간 경계에 있지 않습니다.")
        if settings.equation == "cylinder_ns" and not self.contract.get("provenance"):
            raise ValueError(
                "원통 후류 데이터의 출처·사용 조건을 provenance에 기록하세요."
            )
        self.fingerprint = {
            "mode": settings.data_mode,
            "contract": self.contract,
            "files": self.file_hashes,
        }

    @property
    def residual_names(self):
        return (
            ["f_u", "f_v", "f_div"]
            if len(self.spec["outputs"]) == 3
            else ["f_u", "f_v"] if self.settings.equation == "elasticity2d" else ["f_u"]
        )

    @property
    def fixed_coefficient(self):
        return float(self.contract.get("fixed_coefficient", self.spec["fixed"]))

    @property
    def truth(self):
        return (
            self.spec["fixed"]
            if self.settings.data_mode == "synthetic"
            else self.contract.get("reference_coefficient")
        )

    def labels(self, x):
        if self.settings.data_mode != "synthetic":
            raise ValueError("실데이터 모드에서 합성 관측 생성 금지")
        return exact_field(self.settings.equation, x)

    def forcing(self, x):
        if self.settings.data_mode == "synthetic":
            return known_forcing(self.settings.equation, x)
        if self.contract["forcing_mode"] == "zero":
            return np.zeros((len(x), len(self.residual_names)))
        from scipy.interpolate import LinearNDInterpolator

        if not hasattr(self, "_forcing"):
            px, py = self.external["forcing"]
            if not np.isfinite(py).all():
                raise ValueError("모든 외력 성분이 필요합니다.")
            self._forcing = LinearNDInterpolator(px, py)
        values = np.asarray(self._forcing(x))
        if not np.isfinite(values).all():
            raise ValueError(
                "외력 CSV의 보간 영역 밖입니다. 영역 전체를 덮는 알려진 외력이 필요합니다."
            )
        return values

    def scales(self):
        if self.settings.data_mode == "csv":
            return {
                "field": self.contract["field_scales"],
                "residual": self.contract["residual_scales"],
                "provenance": "explicit canonical-units data contract",
            }
        # Prescribed benchmark characteristic scales, not sampled holdout labels.
        characteristic = {
            "reaction_diffusion": [0.2],
            "elasticity2d": [1.0, 0.5],
            "kovasznay": [2.0, 1.0, 1.0],
            "taylor_green": [1.0, 1.0, 0.5],
        }
        field = np.asarray(
            characteristic.get(
                self.settings.equation, [1.0] * len(self.spec["outputs"])
            )
        )
        length = min(
            self.domain[a][1] - self.domain[a][0]
            for a in range(len(self.domain))
            if a != self.spec["time_axis"]
        )
        return {
            "field": field.tolist(),
            "residual": [float(max(field.max() / min(length, length**2), 0.001))]
            * len(self.residual_names),
            "provenance": "prescribed synthetic characteristic field/domain scales; no holdout labels read",
        }

    def training(self, plan, cycle):
        s = self.settings
        arrays = {}
        if self.external:
            for family in ("observations", "boundary", "initial", "gauge"):
                if family in self.external:
                    x, y = self.external[family]
                    mask = plan.training_mask(x, family)
                    if not mask.any():
                        raise ValueError(family + " 학습점이 없습니다.")
                    arrays[family] = x[mask].copy()
                    arrays["label_" + family] = y[mask].copy()
            if "initial_velocity" in self.external:
                x, y = self.external["initial_velocity"]
                mask = plan.training_mask(x, "initial")
                arrays.update(
                    initial_velocity_x=x[mask].copy(), initial_velocity=y[mask].copy()
                )
            if (
                plan.mode == "interpolation"
                and plan.training_mask(self.external["observations"][0]).all()
            ):
                raise ValueError("제외 구역에 실제 관측이 없습니다.")
        else:
            x = permitted_lhs(
                plan, s.observation_count, plan.seed, "observations", cycle
            )
            y = self.labels(x)
            if s.noise_fraction:
                y = (
                    y
                    + rng_for(plan.seed, "noise", cycle).normal(size=y.shape)
                    * np.asarray(self.scales()["field"])
                    * s.noise_fraction
                )
            arrays.update(observations=x, label_observations=y)
            spatial = plan.spatial_axes
            boundaries = []
            for a in spatial:
                for end in (0, 1):
                    b = permitted_lhs(
                        plan,
                        s.boundary_count,
                        plan.seed,
                        "boundary",
                        cycle + 100 * a + end,
                    )
                    b[:, a] = plan.domain[a][end]
                    boundaries.append(b)
            b = np.concatenate(boundaries)
            arrays.update(boundary=b, label_boundary=self.labels(b))
            if plan.time_axis is not None:
                x = permitted_lhs(plan, s.initial_count, plan.seed, "initial", cycle)
                x[:, plan.time_axis] = plan.domain[plan.time_axis][0]
                arrays.update(initial=x, label_initial=self.labels(x))
                if s.equation == "wave2d":
                    arrays["initial_velocity"] = np.zeros((len(x), 1))
            if len(self.spec["outputs"]) == 3:
                x = permitted_lhs(plan, 16, plan.seed, "gauge", cycle)
                x[:, plan.spatial_axes] = np.array(plan.domain)[
                    list(plan.spatial_axes), 0
                ]
                arrays.update(gauge=x, label_gauge=self.labels(x))
        for family in ("observations", "boundary", "initial", "gauge"):
            if family in arrays:
                plan.assert_training(arrays[family], family)
        return arrays

    def validation(self, plan):
        if self.external:
            x, y = self.external["validation"]
            mask = plan.validation_mask(x)
            if not mask.any():
                raise ValueError("허용 학습 영역에 중단 검증점이 없습니다.")
            return {"x": x[mask].copy(), "y": y[mask].copy()}
        x = permitted_lhs(
            plan, self.settings.validation_count, 991823, "observations", 0
        )
        return {"x": x, "y": self.labels(x)}

    def evaluation(self, plan):
        if self.external:
            if "evaluation" not in self.external:
                return None
            x, y = self.external["evaluation"]
            mask = plan.evaluation_mask(x)
            if not mask.any():
                raise ValueError("평가 영역에 별도 관측이 없습니다.")
            return {"x": x[mask].copy(), "y": y[mask].copy()}
        if plan.mode == "basic":
            x = lhs(
                plan.domain,
                self.settings.evaluation_count,
                rng_for(93235, "evaluation"),
            )
        else:
            x = plan.fixed_evaluation_points(per_spatial_axis=16, time_count=9)
        return {"x": x, "y": self.labels(x)}


In [ ]:
%%writefile "{V4_MODULE_DIR}/diagnostics.py"
"""Read-only diagnostics using the actual float32 residual for gradient checks.

Installed during training by progress.training_progress. The numerical runtime
source and checkpoint identity are unchanged; no downloaded source is executed.
"""

import numpy as np

from .config import group_weights, learning_rates
from .equations import residual

REVISION = "coefficient-observer-v2"


def coefficient_audit(runtime, tensors, gradients):
    """Check d(MSE)/dc against autodiff without subtracting float32 residuals.

    Spatial derivatives are frozen. A float64 affine slope is evaluated from the
    declared equation; its product with the actual float32 normalized residual
    matches the loss's computational path. The separate float64 reconstruction
    remains available for the frozen-field optimum and roundoff diagnostics.
    """
    tf = runtime.tf
    q, g, h = runtime.derivatives(tensors["pde"])
    coefficient = float(runtime.coefficient)
    scale32 = np.asarray(runtime.scales["residual"], dtype=np.float32)
    actual = residual(
        runtime.settings.equation,
        tensors["pde"],
        q,
        g,
        h,
        runtime.coefficient,
        tensors["forcing"],
        tf,
    ) / tf.constant(scale32)
    actual = actual.numpy().astype(np.float64)
    arguments = [
        runtime.settings.equation,
        tensors["pde"].numpy().astype(np.float64),
        q.numpy().astype(np.float64),
        g.numpy().astype(np.float64),
        h.numpy().astype(np.float64),
    ]
    forcing = tensors["forcing"].numpy().astype(np.float64)
    scale = scale32.astype(np.float64)
    intercept = residual(*arguments, 0.0, forcing, np) / scale
    slope = residual(*arguments, 1.0, forcing, np) / scale - intercept
    values = np.asarray(gradients, dtype=np.float64)
    if not all(np.isfinite(v).all() for v in (actual, intercept, slope, values)):
        raise RuntimeError("계수 gradient 검사에 유한하지 않은 값이 있습니다.")
    products = 2.0 * actual * slope
    analytic = float(np.mean(products))
    # Bound reduction/chain-rule rounding by the magnitude before cancellation.
    rounding_allowance = float(
        32 * np.finfo(np.float32).eps * np.mean(np.abs(products))
    )
    tolerance = max(1e-5, 1e-3 * abs(values[0]), rounding_allowance)
    error = abs(analytic - values[0])
    matches = bool(error <= tolerance)
    independent = bool(np.allclose(values[1:], 0.0, rtol=0, atol=1e-8))
    if not matches or not independent:
        raise RuntimeError(
            "계수 gradient 의존성 검사 실패: "
            f"analytic={analytic:.9g}, autodiff={values[0]:.9g}, "
            f"difference={error:.9g}, tolerance={tolerance:.9g}, "
            f"data_constraint_independent={independent}"
        )
    denominator = float(np.mean(slope * slope))
    optimum = (
        -float(np.mean(intercept * slope)) / denominator
        if denominator > 1e-20
        else None
    )
    reconstructed = intercept + coefficient * slope
    return {
        "unconstrained_optimum": optimum,
        "positive_domain_infimum": max(0.0, optimum) if optimum is not None else None,
        "analytic_gradient": analytic,
        "gradient_matches_autodiff": matches,
        "gradient_absolute_difference": error,
        "gradient_absolute_tolerance": tolerance,
        "float32_rounding_allowance": rounding_allowance,
        "reference_float64_gradient": float(2 * np.mean(reconstructed * slope)),
        "residual_roundoff_max_abs": float(np.max(np.abs(actual - reconstructed))),
        "data_constraint_coefficient_independent": independent,
        "field_frozen_only": True,
        "observer_revision": REVISION,
    }


def observe(runtime, tensors, iteration):
    """Observe losses/gradients while retaining model, optimizer and RNG guards."""
    before = runtime.snapshot()
    try:
        terms, vectors, coefficient_gradients = runtime.term_gradients(tensors)
        norms = np.array([np.linalg.norm(v) for v in vectors])
        cosine = np.diag((norms > 1e-30).astype(float))
        for a in range(3):
            for b in range(a):
                cosine[a, b] = cosine[b, a] = float(
                    vectors[a] @ vectors[b] / max(norms[a] * norms[b], 1e-30)
                )
        frozen = (
            coefficient_audit(runtime, tensors, coefficient_gradients)
            if runtime.inverse
            else None
        )
        return {
            "iteration": iteration,
            "raw_normalized_groups": [float(v) for v in terms],
            "network_gradient_norms": norms.tolist(),
            "gradient_cosines": cosine.tolist(),
            "coefficient_term_gradients": coefficient_gradients,
            "coefficient": float(runtime.coefficient),
            "frozen_field_coefficient": frozen,
            "gradient_cosine_defined": (
                (norms > 1e-30)[:, None] & (norms > 1e-30)[None, :]
            ).tolist(),
            "rng_guard_scope": "Python, NumPy legacy RNG, TensorFlow global Generator; deterministic tanh model",
            "effective_weights": group_weights(
                runtime.arm, iteration, runtime.factors
            ).tolist(),
            "sa_factors": runtime.factors.tolist(),
            "next_learning_rates": list(
                learning_rates(iteration, runtime.settings.base_lr)
            ),
            "state_guard": "PASS",
            "observer_revision": REVISION,
            "not_a_causal_proof": True,
        }
    finally:
        runtime.assert_unchanged(before)


In [ ]:
%%writefile "{V4_MODULE_DIR}/equations.py"
"""Manufactured benchmarks with explicit known forcing, separate from inference.

Truth constants below belong to synthetic data generation/auditing, never to the
learned coefficient initializer. Real-data forcing is supplied by its contract.
"""

import numpy as np

EQUATIONS = {
    "wave2d": dict(
        label="2차원 파동",
        domain=[[0, 1]] * 3,
        axes=["x", "y", "t"],
        outputs=["u"],
        time_axis=2,
        coefficient="lambda1",
        fixed=1.0,
        initial=0.5,
    ),
    "heat2d": dict(
        label="2차원 열전도",
        domain=[[0, 1]] * 3,
        axes=["x", "y", "t"],
        outputs=["u"],
        time_axis=2,
        coefficient="alpha",
        fixed=0.1,
        initial=0.05,
    ),
    "poisson2d": dict(
        label="2차원 포아송",
        domain=[[0, 1]] * 2,
        axes=["x", "y"],
        outputs=["u"],
        time_axis=None,
        coefficient="source_amplitude",
        fixed=1.0,
        initial=0.5,
    ),
    "burgers": dict(
        label="버거스",
        domain=[[-1, 1], [0, 1]],
        axes=["x", "t"],
        outputs=["u"],
        time_axis=1,
        coefficient="nu",
        fixed=0.01 / np.pi,
        initial=0.0015,
    ),
    "kovasznay": dict(
        label="Navier–Stokes · Kovasznay",
        domain=[[-0.5, 1], [-0.5, 1.5]],
        axes=["x", "y"],
        outputs=["u", "v", "p"],
        time_axis=None,
        coefficient="nu",
        fixed=0.025,
        initial=0.0125,
    ),
    "taylor_green": dict(
        label="Navier–Stokes · Taylor–Green",
        domain=[[-np.pi, np.pi], [-np.pi, np.pi], [0, 1]],
        axes=["x", "y", "t"],
        outputs=["u", "v", "p"],
        time_axis=2,
        coefficient="nu",
        fixed=0.01,
        initial=0.005,
    ),
    "darcy2d": dict(
        label="2차원 Darcy",
        domain=[[0, 1]] * 2,
        axes=["x", "y"],
        outputs=["u"],
        time_axis=None,
        coefficient="permeability_multiplier",
        fixed=1.0,
        initial=0.5,
    ),
    "reaction_diffusion": dict(
        label="반응–확산",
        domain=[[0, 1]] * 3,
        axes=["x", "y", "t"],
        outputs=["u"],
        time_axis=2,
        coefficient="D",
        fixed=0.01,
        initial=0.005,
    ),
    "advection_diffusion": dict(
        label="이류–확산",
        domain=[[0, 1]] * 3,
        axes=["x", "y", "t"],
        outputs=["u"],
        time_axis=2,
        coefficient="D",
        fixed=0.1,
        initial=0.05,
    ),
    "helmholtz": dict(
        label="Helmholtz",
        domain=[[0, 1]] * 2,
        axes=["x", "y"],
        outputs=["u"],
        time_axis=None,
        coefficient="wavenumber_squared",
        fixed=1.0,
        initial=0.5,
    ),
    "elasticity2d": dict(
        label="2차원 선형탄성 · 평면변형률",
        domain=[[0, 1]] * 2,
        axes=["x", "y"],
        outputs=["u", "v"],
        time_axis=None,
        coefficient="young_modulus",
        fixed=1.0,
        initial=0.5,
    ),
    "cylinder_ns": dict(
        label="Navier–Stokes · 원통 후류 (외부 데이터)",
        domain=[[1, 8], [-2, 2], [0, 1]],
        axes=["x", "y", "t"],
        outputs=["u", "v", "p"],
        time_axis=2,
        coefficient="nu",
        fixed=0.01,
        initial=0.005,
    ),
}


def exact_field(name, z, xp=np):
    x, y = z[:, 0], z[:, 1]
    pi = np.pi
    if name == "cylinder_ns":
        raise ValueError("원통 후류에는 해석 정답을 생성하지 않습니다.")
    if name == "burgers":
        return (0.5 - 0.5 * xp.tanh((x - 0.5 * y) / (4 * (0.01 / pi))))[:, None]
    if name == "kovasznay":
        lam = 20 - np.sqrt(400 + 4 * pi * pi)
        e = xp.exp(lam * x)
        return xp.stack(
            [
                1 - e * xp.cos(2 * pi * y),
                lam / (2 * pi) * e * xp.sin(2 * pi * y),
                0.5 * (1 - e**2),
            ],
            axis=1,
        )
    if name == "taylor_green":
        e = xp.exp(-0.02 * z[:, 2])
        return xp.stack(
            [
                -xp.cos(x) * xp.sin(y) * e,
                xp.sin(x) * xp.cos(y) * e,
                -0.25 * (xp.cos(2 * x) + xp.cos(2 * y)) * e**2,
            ],
            axis=1,
        )
    s = xp.sin(pi * x) * xp.sin(pi * y)
    if name == "elasticity2d":
        return xp.stack([s, 0.5 * s], axis=1)
    if name in {"poisson2d", "darcy2d", "helmholtz"}:
        return s[:, None]
    if name == "wave2d":
        return (s * xp.cos(np.sqrt(2) * pi * z[:, 2]))[:, None]
    if name == "heat2d":
        return (s * xp.exp(-0.2 * pi * pi * z[:, 2]))[:, None]
    if name == "reaction_diffusion":
        return (0.2 * s * xp.exp(-z[:, 2]))[:, None]
    if name == "advection_diffusion":
        return (s * xp.exp(-z[:, 2]))[:, None]
    raise ValueError(name)


def known_forcing(name, z, xp=np):
    x, y = z[:, 0], z[:, 1]
    s = xp.sin(np.pi * x) * xp.sin(np.pi * y)
    zero = x * 0
    if name in {"kovasznay", "taylor_green", "cylinder_ns"}:
        return xp.stack([zero, zero, zero], axis=1)
    if name == "elasticity2d":
        nu = 0.25  # Known Poisson ratio, not inferred jointly with E.
        lam, mu = nu / ((1 + nu) * (1 - 2 * nu)), 1 / (2 * (1 + nu))
        mixed = np.pi**2 * xp.cos(np.pi * x) * xp.cos(np.pi * y)
        return xp.stack(
            [
                (lam + 3 * mu) * np.pi**2 * s - 0.5 * (lam + mu) * mixed,
                0.5 * (lam + 3 * mu) * np.pi**2 * s - (lam + mu) * mixed,
            ],
            axis=1,
        )
    if name == "poisson2d":
        return (2 * np.pi**2 * s)[:, None]
    if name == "helmholtz":
        return ((2 * np.pi**2 - 1) * s)[:, None]
    if name == "darcy2d":
        sx, sy = xp.sin(np.pi * x), xp.sin(np.pi * y)
        cx, cy = xp.cos(np.pi * x), xp.cos(np.pi * y)
        k = 1 + 0.5 * s
        return (
            2 * np.pi**2 * k * s - 0.5 * np.pi**2 * ((cx * sy) ** 2 + (sx * cy) ** 2)
        )[:, None]
    if name == "reaction_diffusion":
        value = 0.2 * s * xp.exp(-z[:, 2])
        return ((-2 + 0.02 * np.pi**2) * value + value**3)[:, None]
    if name == "advection_diffusion":
        e = xp.exp(-z[:, 2])
        return (
            (-1 + 0.2 * np.pi**2) * s * e
            + 0.5 * np.pi * xp.cos(np.pi * x) * xp.sin(np.pi * y) * e
            + 0.25 * np.pi * xp.sin(np.pi * x) * xp.cos(np.pi * y) * e
        )[:, None]
    return zero[:, None]


def residual(name, x, q, g, h, coefficient, forcing, xp=np):
    """q(N,C), g(N,C,D), full Hessian h(N,C,D,D); all forcing is known input."""
    u, ux, uxx = q[:, 0], g[:, 0, 0], h[:, 0, 0, 0]
    if name == "burgers":
        return (g[:, 0, 1] + u * ux - coefficient * uxx)[:, None] - forcing
    uy, lap = g[:, 0, 1], uxx + h[:, 0, 1, 1]
    if name in {"kovasznay", "taylor_green", "cylinder_ns"}:
        v = q[:, 1]
        ut = g[:, 0, 2] if name != "kovasznay" else u * 0
        vt = g[:, 1, 2] if name != "kovasznay" else u * 0
        return (
            xp.stack(
                [
                    ut + u * ux + v * uy + g[:, 2, 0] - coefficient * lap,
                    vt
                    + u * g[:, 1, 0]
                    + v * g[:, 1, 1]
                    + g[:, 2, 1]
                    - coefficient * (h[:, 1, 0, 0] + h[:, 1, 1, 1]),
                    ux + g[:, 1, 1],
                ],
                axis=1,
            )
            - forcing
        )
    if name == "elasticity2d":
        nu = 0.25
        lam = coefficient * nu / ((1 + nu) * (1 - 2 * nu))
        mu = coefficient / (2 * (1 + nu))
        return (
            xp.stack(
                [
                    -(lam + 2 * mu) * uxx
                    - mu * h[:, 0, 1, 1]
                    - (lam + mu) * h[:, 1, 0, 1],
                    -mu * h[:, 1, 0, 0]
                    - (lam + 2 * mu) * h[:, 1, 1, 1]
                    - (lam + mu) * h[:, 0, 0, 1],
                ],
                axis=1,
            )
            - forcing
        )
    if name == "wave2d":
        return (h[:, 0, 2, 2] - coefficient * lap)[:, None] - forcing
    if name == "heat2d":
        return (g[:, 0, 2] - coefficient * lap)[:, None] - forcing
    if name == "poisson2d":
        return -lap[:, None] - coefficient * forcing
    if name == "helmholtz":
        return (-lap - coefficient * u)[:, None] - forcing
    if name == "darcy2d":
        sx, sy = xp.sin(np.pi * x[:, 0]), xp.sin(np.pi * x[:, 1])
        cx, cy = xp.cos(np.pi * x[:, 0]), xp.cos(np.pi * x[:, 1])
        return (
            -coefficient
            * (
                (1 + 0.5 * sx * sy) * lap
                + 0.5 * np.pi * cx * sy * ux
                + 0.5 * np.pi * sx * cy * uy
            )
        )[:, None] - forcing
    if name == "reaction_diffusion":
        return (g[:, 0, 2] - coefficient * lap - u + u**3)[:, None] - forcing
    if name == "advection_diffusion":
        return (g[:, 0, 2] + 0.5 * ux + 0.25 * uy - coefficient * lap)[
            :, None
        ] - forcing
    raise ValueError(name)


In [ ]:
%%writefile "{V4_MODULE_DIR}/experiments.py"
"""Explicit batch plans and sensitivity cases; no automatic optimization search."""

from dataclasses import replace
from itertools import product
import json
from pathlib import Path
from .config import digest
from .data import DataSource
from .equations import EQUATIONS
from .runner import run_all
from .storage import write_json


def experiment_plan(
    base, equations=None, directions=(False,), noise=(0.0,), initial_multipliers=(1.0,)
):
    base.validate()
    equations = equations or (base.equation,)
    configurations = []
    if base.data_mode == "csv" and tuple(equations) != (base.equation,):
        raise ValueError("CSV 계획은 계약에 지정한 방정식만 사용합니다.")
    if base.data_mode == "csv" and noise != (0.0,):
        raise ValueError(
            "실제 관측은 파일 값을 유지합니다. 노이즈 대조 파일은 별도 계약으로 연결하세요."
        )
    for eq, inverse, level, multiplier in product(
        equations, directions, noise, initial_multipliers
    ):
        if not inverse and multiplier != 1.0:
            continue
        initial = (
            base.initial_coefficient
            if base.initial_coefficient is not None
            else EQUATIONS[eq]["initial"]
        ) * multiplier
        setting = replace(
            base,
            equation=eq,
            inverse=inverse,
            noise_fraction=level,
            initial_coefficient=initial if inverse else None,
        ).validate()
        configurations.append(setting)
    keys = [digest(s.to_dict()) for s in configurations]
    if len(set(keys)) != len(keys):
        raise ValueError("계획에 같은 실험 조건이 중복되었습니다.")
    if not configurations or len(configurations) > 200:
        raise ValueError("계획 조건 수는 1~200입니다.")
    return configurations


def execute_plan(
    root, configurations, results_zip, resume_zip, training_authorized=False, log=print
):
    if training_authorized is not True:
        raise PermissionError("학습 계획 실행 허용이 필요합니다.")
    root = Path(root)
    plan = [s.to_dict() for s in configurations]
    path = root / "plan.json"
    if path.exists() and json.loads(path.read_text(encoding="utf-8")) != json.loads(
        json.dumps(plan)
    ):
        raise ValueError("재개 계획이 저장된 계획과 다릅니다.")
    write_json(path, plan)
    reports = []
    for settings in configurations:
        source = DataSource(
            settings, root / "inputs/data.json" if settings.data_mode == "csv" else None
        )
        reports.extend(
            run_all(root, settings, source, results_zip, resume_zip, True, log)
        )
    return reports


In [ ]:
%%writefile "{V4_MODULE_DIR}/future_evaluation.py"
"""Evaluate frozen basic forward models beyond their original time domain."""

import csv
import io
import json
from pathlib import Path
import zipfile

import numpy as np

from .config import Settings
from .data import DataSource
from .equations import EQUATIONS, exact_field
from .metrics import observed_metrics
from .reporting import reports
from .runtime import Runtime
from .storage import sha_file


def future_grid(spec, steps=20):
    """Future time grid at 1% increments of the original time span."""
    if steps not in (20, 200):
        raise ValueError("평가 시점 수는 20 또는 200입니다.")
    time_axis = spec["time_axis"]
    axes = []
    spatial_count = 256 if len(spec["domain"]) == 2 else 32
    for axis, (low, high) in enumerate(spec["domain"]):
        if axis == time_axis:
            values = high + (high - low) * np.arange(1, steps + 1) / 100
        else:
            values = (
                low + (high - low) * (np.arange(spatial_count) + 0.5) / spatial_count
            )
        axes.append(values)
    return np.stack(np.meshgrid(*axes, indexing="ij"), axis=-1).reshape(-1, len(axes))


def evaluate_saved(root, log=print, *, steps=20):
    root = Path(root)
    run_id = json.loads((root / "contract.json").read_text(encoding="utf-8"))["run_id"]
    rows = []
    for report in reports(root):
        equation = report["equation"]
        spec = EQUATIONS[equation]
        if (
            spec["time_axis"] is None
            or report["inverse"]
            or report["execution_scope"] != "research"
            or report["generalization"]["mode"] != "basic"
        ):
            continue
        trial = root / "trials" / report["trial_id"]
        cfg = json.loads((trial / "trial_settings.json").read_text(encoding="utf-8"))
        settings = Settings.from_dict(cfg["settings"])
        if settings.data_mode != "synthetic" or settings.noise_fraction:
            continue
        source = DataSource(settings)
        runtime = Runtime(settings, report["arm"], source)
        points = future_grid(spec, steps)
        reference = exact_field(equation, points)
        times = points[:, spec["time_axis"]]
        sample_times = np.unique(times)
        last = times == sample_times[-1]
        with np.load(trial / "fixed.npz", allow_pickle=False) as fixed:
            check_x, check_y = fixed["evaluation_x"], fixed["evaluation_y"]
        for stage in ("adam", "lbfgs"):
            if report.get(stage) is None:
                continue
            checkpoint = trial / (stage + ".npz")
            runtime.load(checkpoint)
            replay = observed_metrics(check_y, runtime.predict(check_x))
            original = report[stage]["field"]
            for metric in ("relative_l1", "relative_l2"):
                if not np.isclose(
                    replay[metric], original[metric], rtol=0.002, atol=0.00002
                ):
                    raise ValueError(
                        f"저장 평가 재현 오차: {trial.name} {stage} {metric}"
                    )
            prediction = runtime.predict(points, batch=2048)
            future = observed_metrics(reference, prediction)
            endpoint = observed_metrics(reference[last], prediction[last])
            windows = {}
            for start in range(0, steps, 20):
                lo, hi = sample_times[start], sample_times[start + 19]
                mask = (times >= lo) & (times <= hi)
                block = observed_metrics(reference[mask], prediction[mask])
                for norm in ("l1", "l2"):
                    windows[f"t{lo:.2f}_{hi:.2f}_{norm}_pct"] = (
                        100 * block["relative_" + norm]
                    )
            rows.append(
                dict(
                    run_id=run_id,
                    trial_id=trial.name,
                    equation=equation,
                    arm=report["arm"],
                    seed=report["seed"],
                    stage=stage,
                    checkpoint_sha256=sha_file(checkpoint),
                    points=len(points),
                    original_l1_pct=100 * original["relative_l1"],
                    original_l2_pct=100 * original["relative_l2"],
                    future_l1_pct=100 * future["relative_l1"],
                    future_l2_pct=100 * future["relative_l2"],
                    endpoint_l1_pct=100 * endpoint["relative_l1"],
                    endpoint_l2_pct=100 * endpoint["relative_l2"],
                    **windows,
                )
            )
        log(f"{spec['label']} · {report['arm']} · {report['seed']} 평가 완료")
    if not rows:
        raise ValueError(
            "합성 기본 비교의 무노이즈 순방향 시간 의존 모델을 포함한 결과 ZIP을 선택하세요."
        )
    return rows


def save_evaluation(rows, path, *, steps=20):
    end = 1 + steps / 100
    if f"t{end - .19:.2f}_{end:.2f}_l2_pct" not in rows[0]:
        raise ValueError("시간 범위와 결과 열 구성이 다릅니다.")
    table = io.StringIO(newline="")
    writer = csv.DictWriter(table, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
    protocol = dict(
        training_updates=0,
        forecast_fraction=steps / 100,
        time_grid=f"1.01 to {end:.2f} in steps of 0.01 for training domain [0,1]",
        window_columns="non-overlapping 20-sample intervals, inclusive of both labelled times",
        spatial_grid="cell centres: 256 in 1D, 32 x 32 in 2D",
        reference="continued synthetic exact_field",
        scaling="original training domain and field scales",
        metrics="relative errors in percent, maximum per output component",
        endpoint="last future time only",
        stage_rows=len(rows),
    )
    with zipfile.ZipFile(path, "x", zipfile.ZIP_DEFLATED) as bundle:
        bundle.writestr("comparison.csv", table.getvalue().encode("utf-8-sig"))
        bundle.writestr("protocol.json", json.dumps(protocol, indent=2))


In [ ]:
%%writefile "{V4_MODULE_DIR}/generalization.py"
"""Independent interpolation and temporal-extrapolation dataset contracts.

No TensorFlow, model loading, label generation or training occurs here. The V4
runner must use these contracts on every initial batch and resampling cycle.
"""

from dataclasses import dataclass
import hashlib
import json

import numpy as np

MODES = {"basic", "interpolation", "time_extrapolation"}
FAMILIES = {"observations", "pde", "boundary", "initial", "gauge"}
SEED_ORIGIN = 3234
BLOCK_COUNT = 16


@dataclass(frozen=True)
class GeneralizationPlan:
    """Immutable split, shared by methods and independent of sampling cycles.

    Spatial block numbering is row-major: x changes fastest in 2D. Edges belong
    to the block on their positive side; the domain maximum is in the last block.
    Known initial/boundary/gauge constraints remain available in interpolation.
    """

    domain: tuple
    spatial_axes: tuple
    time_axis: int | None = None
    mode: str = "basic"
    seed: int = SEED_ORIGIN
    block_override: int | None = None
    time_train_fraction: float = 0.8
    version: str = "v4-generalization-16-v1"

    def __post_init__(self):
        domain = np.asarray(self.domain, dtype=np.float64)
        if domain.ndim != 2 or domain.shape[1] != 2 or not len(domain):
            raise ValueError("domain must contain [minimum, maximum] per axis")
        if not np.isfinite(domain).all() or np.any(domain[:, 0] >= domain[:, 1]):
            raise ValueError("Domain limits must be finite and increasing")
        axes = tuple(self.spatial_axes)
        if len(axes) not in (1, 2) or any(type(a) is not int for a in axes):
            raise ValueError("Exactly one or two integer spatial axes are supported")
        if len(set(axes)) != len(axes) or any(a < 0 or a >= len(domain) for a in axes):
            raise ValueError("Invalid spatial axes")
        if self.time_axis is not None and (
            type(self.time_axis) is not int
            or self.time_axis in axes
            or self.time_axis < 0
            or self.time_axis >= len(domain)
        ):
            raise ValueError("Invalid time axis")
        covered = set(axes) | (
            {self.time_axis} if self.time_axis is not None else set()
        )
        if covered != set(range(len(domain))):
            raise ValueError("Every coordinate axis must be spatial or temporal")
        if self.mode not in MODES or type(self.seed) is not int or self.seed < 0:
            raise ValueError("Invalid mode or seed")
        if self.version != "v4-generalization-16-v1":
            raise ValueError("Unsupported split version")
        if self.block_override is not None and (
            type(self.block_override) is not int or not 0 <= self.block_override < 16
        ):
            raise ValueError("block_override must be an integer in 0..15")
        if (
            not np.isfinite(self.time_train_fraction)
            or not 0 < self.time_train_fraction < 1
        ):
            raise ValueError("time_train_fraction must be between zero and one")
        if self.mode == "time_extrapolation" and self.time_axis is None:
            raise ValueError(
                "Temporal extrapolation requires a time-dependent equation"
            )
        object.__setattr__(self, "domain", tuple(tuple(map(float, r)) for r in domain))
        object.__setattr__(self, "spatial_axes", axes)

    @property
    def block_id(self):
        if self.block_override is not None:
            return self.block_override
        # Coprime stride visits every block once over 16 consecutive seeds.
        return (5 * (self.seed - SEED_ORIGIN) + 7) % BLOCK_COUNT

    @property
    def cutoff(self):
        if self.time_axis is None:
            return None
        lo, hi = self.domain[self.time_axis]
        return lo + self.time_train_fraction * (hi - lo)

    @property
    def training_domain(self):
        """Sampling bounds for LHS, RAD candidate generation and BC sampling."""
        bounds = [list(row) for row in self.domain]
        if self.mode == "time_extrapolation":
            bounds[self.time_axis][1] = self.cutoff
        return bounds

    def coordinates(self, values):
        points = np.asarray(values, dtype=np.float64)
        if points.ndim != 2 or points.shape[1] != len(self.domain):
            raise ValueError("Coordinates must be an N by domain-dimension array")
        bounds = np.asarray(self.domain)
        if not np.isfinite(points).all():
            raise ValueError("Coordinates must be finite")
        if np.any(points < bounds[:, 0]) or np.any(points > bounds[:, 1]):
            raise ValueError("Coordinates outside the declared domain")
        return points

    def spatial_block_ids(self, values):
        points = self.coordinates(values)
        divisions = 16 if len(self.spatial_axes) == 1 else 4
        ids = np.zeros(len(points), dtype=np.int64)
        for position, axis in enumerate(self.spatial_axes):
            lo, hi = self.domain[axis]
            unit = (points[:, axis] - lo) / (hi - lo)
            cells = np.minimum(
                np.floor(unit * divisions).astype(np.int64), divisions - 1
            )
            ids += cells * divisions**position
        return ids

    def training_mask(self, values, family="observations"):
        """Apply on every renewal. Initial/BC constraints are not held-out labels."""
        if family not in FAMILIES:
            raise ValueError("Unknown training family")
        points = self.coordinates(values)
        if self.mode == "time_extrapolation":
            return points[:, self.time_axis] <= self.cutoff
        if self.mode == "interpolation" and family == "observations":
            return self.spatial_block_ids(points) != self.block_id
        return np.ones(len(points), dtype=bool)

    def evaluation_mask(self, values):
        points = self.coordinates(values)
        if self.mode == "interpolation":
            return self.spatial_block_ids(points) == self.block_id
        if self.mode == "time_extrapolation":
            return points[:, self.time_axis] > self.cutoff
        return np.ones(len(points), dtype=bool)

    def split_observations(self, coordinates, labels):
        """Return aligned copies; never discard or alter the source dataset."""
        points = self.coordinates(coordinates)
        labels = np.asarray(labels)
        if labels.ndim not in (1, 2) or len(labels) != len(points):
            raise ValueError("Labels must align with coordinates")
        if not np.issubdtype(labels.dtype, np.number) or not np.isfinite(labels).all():
            raise ValueError("Labels must be finite numeric values")
        mask = self.training_mask(points)
        if not mask.any():
            raise ValueError("No training observations remain")
        if self.mode != "basic" and mask.all():
            raise ValueError(
                "No held-out observations exist; revise the observation layout"
            )
        return {
            "train_coordinates": points[mask].copy(),
            "train_labels": labels[mask].copy(),
            "heldout_coordinates": points[~mask].copy(),
            "heldout_labels": labels[~mask].copy(),
            "train_mask": mask.copy(),
        }

    def assert_training(self, coordinates, family="observations"):
        """Fail closed if any training/resampled point violates the split."""
        if not self.training_mask(coordinates, family).all():
            raise ValueError(
                "Training point leaks into the held-out observation/time region"
            )

    def validation_mask(self, coordinates, family="observations"):
        """Stopping/tuning validation uses only the permitted training region."""
        return self.training_mask(coordinates, family)

    def fingerprint(self):
        return hashlib.sha256(
            json.dumps(self.to_dict(), sort_keys=True, separators=(",", ":")).encode()
        ).hexdigest()

    def to_dict(self):
        return {
            "version": self.version,
            "domain": [list(row) for row in self.domain],
            "spatial_axes": list(self.spatial_axes),
            "time_axis": self.time_axis,
            "mode": self.mode,
            "seed": self.seed,
            "block_override": self.block_override,
            "time_train_fraction": float(self.time_train_fraction),
        }

    @classmethod
    def from_dict(cls, value):
        return cls(**value)

    def summary(self):
        return {
            **self.to_dict(),
            "fingerprint": self.fingerprint(),
            "block_count": BLOCK_COUNT if self.mode == "interpolation" else None,
            "heldout_block": self.block_id if self.mode == "interpolation" else None,
            "training_domain": self.training_domain,
            "time_cutoff": self.cutoff if self.mode == "time_extrapolation" else None,
            "heldout_used_for_stopping": False,
            "independent_training_required": self.mode != "basic",
        }

    def fixed_evaluation_points(self, per_spatial_axis=32, time_count=21):
        """Deterministic cell-centre grid; never uses or advances training RNG.

        Interpolation grids require multiples of 4 (2D) or 16 (1D), ensuring
        every block has the same number of evaluation points.
        """
        divisions = 16 if len(self.spatial_axes) == 1 else 4
        if type(per_spatial_axis) is not int or per_spatial_axis < divisions:
            raise ValueError("Evaluation resolution is too small")
        if self.mode == "interpolation" and per_spatial_axis % divisions:
            raise ValueError(
                "Interpolation resolution must be divisible by block divisions"
            )
        if type(time_count) is not int or time_count < 1:
            raise ValueError("time_count must be positive")
        axes = []
        for axis, (lo, hi) in enumerate(self.domain):
            if axis == self.time_axis:
                start = self.cutoff if self.mode == "time_extrapolation" else lo
                values = start + (np.arange(time_count) + 0.5) / time_count * (
                    hi - start
                )
            else:
                values = lo + (np.arange(per_spatial_axis) + 0.5) / per_spatial_axis * (
                    hi - lo
                )
            axes.append(values)
        if np.prod([len(a) for a in axes], dtype=np.int64) > 1_000_000:
            raise ValueError("Evaluation grid exceeds one million points")
        points = np.stack(np.meshgrid(*axes, indexing="ij"), axis=-1).reshape(
            -1, len(axes)
        )
        selected = points[self.evaluation_mask(points)]
        if not len(selected):
            raise ValueError("Empty evaluation region")
        return selected.copy()


def field_metrics(reference, prediction):
    """Aggregate relative errors for final reporting, not stopping decisions.

    Returns null relative errors for an all-zero reference instead of adding an
    arbitrary denominator epsilon. Components should also be reported separately
    by the runner when output dimensions have different meanings/scales.
    """
    reference = np.asarray(reference, dtype=np.float64)
    prediction = np.asarray(prediction, dtype=np.float64)
    if (
        reference.shape != prediction.shape
        or reference.ndim not in (1, 2)
        or not reference.size
    ):
        raise ValueError("Reference and prediction need matching non-empty shapes")
    if not np.isfinite(reference).all() or not np.isfinite(prediction).all():
        raise ValueError("Reference and prediction must be finite")
    error = prediction - reference
    l1_denominator = float(np.abs(reference).sum())
    l2_denominator = float(np.linalg.norm(reference))
    return {
        "relative_l1": (
            float(np.abs(error).sum() / l1_denominator) if l1_denominator else None
        ),
        "relative_l2": (
            float(np.linalg.norm(error) / l2_denominator) if l2_denominator else None
        ),
        "mse": float(np.mean(error**2)),
        "relative_errors_defined": bool(l1_denominator and l2_denominator),
        "used_for_stopping": False,
    }


In [ ]:
%%writefile "{V4_MODULE_DIR}/interface.py"
"""Korean bottom panel. Defaults are read-only; no implicit training or installs."""

import json
from pathlib import Path
import tempfile
import uuid
from contextlib import nullcontext

from .config import Settings, ARMS, ARM_LABELS, SEEDS, digest
from .data import DataSource
from .equations import EQUATIONS
from .runner import create_run, run_all
from .experiments import experiment_plan, execute_plan
from .reporting import generate_report, make_plots
from .services import Service
from .storage import pack, restore
from .settings_panel import settings_editor


def panel(folder="/content/drive/MyDrive/pinnV4", show=True):
    import ipywidgets as w
    from IPython.display import display, Markdown

    width = w.Layout(width="100%")
    style = {"description_width": "130px"}

    def dropdown(description, options, value):
        return w.Dropdown(
            description=description,
            options=options,
            value=value,
            layout=width,
            style=style,
        )

    action = dropdown(
        "사용할 기능",
        [
            (v, k)
            for k, v in {
                "results": "저장 결과 보기",
                "future": "저장 모델 시간 외삽 평가",
                "train": "학습 설정·실행",
                "predict": "좌표 예측",
                "ask": "자연어 질문",
                "check": "선택 기능 확인",
                "documents": "문서 생성",
                "plots": "결과 시각화",
                "registry": "추론 모델 관리",
            }.items()
        ],
        "results",
    )
    storage = w.Text(value=folder, description="저장 폴더", layout=width, style=style)
    zip_name = w.Text(
        value="V4_Results.zip", description="통합 ZIP", layout=width, style=style
    )
    connect = w.Button(description="Drive 연결")
    local = w.Button(description="로컬 ZIP 사용")
    equation = dropdown(
        "방정식", [(v["label"], k) for k, v in EQUATIONS.items()], "wave2d"
    )
    inverse = dropdown("문제 방향", [("순방향", False), ("역문제", True)], False)

    class SelectionChecks(w.VBox):
        def __init__(self, title, options, selected):
            self.checks = {
                k: w.Checkbox(
                    description=label,
                    value=k in selected,
                    layout=w.Layout(width="100%"),
                )
                for label, k in options
            }
            super().__init__([w.HTML(f"<b>{title}</b>"), *self.checks.values()])

        @property
        def value(self):
            return tuple(k for k, box in self.checks.items() if box.value)

        @value.setter
        def value(self, values):
            for k, box in self.checks.items():
                box.value = k in values

        @property
        def disabled(self):
            return all(box.disabled for box in self.checks.values())

        @disabled.setter
        def disabled(self, value):
            for box in self.checks.values():
                box.disabled = value

    class OrderedEquationChecks(SelectionChecks):
        def __init__(self, title, options, selected):
            super().__init__(title, options, selected)
            self.available = set(self.checks)
            self.order_list = w.Select(description="실행 순서", layout=width, style=style)
            self.up = w.Button(description="위로")
            self.down = w.Button(description="아래로")
            self.children += (self.order_list, w.HBox([self.up, self.down]))
            self._changing_selection = False
            self.value = selected
            for box in self.checks.values():
                box.observe(self._selection_changed, names="value")
            self.order_list.observe(self._update_buttons, names="value")
            self.up.on_click(lambda _: self._move(-1))
            self.down.on_click(lambda _: self._move(1))

        @property
        def value(self):
            return tuple(self._equation_order)

        @value.setter
        def value(self, values):
            self._equation_order = list(dict.fromkeys(k for k in values if k in self.available))
            self._changing_selection = True
            try:
                SelectionChecks.value.fset(self, self._equation_order)
            finally:
                self._changing_selection = False
            self._refresh_order()

        def set_available(self, equations):
            available = set(equations).intersection(self.checks)
            if available != self.available:
                self.available = available
                self.value = self.value
                for key, box in self.checks.items():
                    box.layout.display = "" if key in available else "none"

        def _selection_changed(self, change):
            if self._changing_selection:
                return
            key = next(k for k, box in self.checks.items() if box is change.owner)
            if change.new:
                self._equation_order.append(key)
            else:
                self._equation_order.remove(key)
            self._refresh_order()

        def _refresh_order(self):
            order = self._equation_order
            selected = self.order_list.value
            self.order_list.options = [
                (f"{i}. {self.checks[k].description}", k) for i, k in enumerate(order, 1)
            ]
            self.order_list.value = selected if selected in order else next(iter(order), None)
            self._update_buttons()

        def _update_buttons(self, change=None):
            order = self._equation_order
            selected = self.order_list.value
            locked = self.disabled or selected not in order
            self.up.disabled = locked or selected == order[0]
            self.down.disabled = locked or selected == order[-1]

        def _move(self, step):
            selected = self.order_list.value
            if self.disabled or selected not in self._equation_order:
                return
            i = self._equation_order.index(selected)
            j = i + step
            if 0 <= j < len(self._equation_order):
                self._equation_order.insert(j, self._equation_order.pop(i))
                self._refresh_order()

        @SelectionChecks.disabled.setter
        def disabled(self, value):
            SelectionChecks.disabled.fset(self, value)
            self.order_list.disabled = value
            self._update_buttons()

    arms = SelectionChecks(
        "비교 방법", [(ARM_LABELS[k], k) for k in ARMS], ("baseline",)
    )
    seeds = dropdown(
        "시드 구성", [("1시드 · 3234", (3234,)), ("4시드 · 3234~3237", SEEDS)], (3234,)
    )
    generalization = dropdown(
        "학습 목표",
        [
            ("기본 비교", "basic"),
            ("16구역 관측 제외 · 보간", "interpolation"),
            ("시간 앞80% 학습 · 외삽", "time_extrapolation"),
        ],
        "basic",
    )
    data_mode = dropdown(
        "데이터 연결",
        [("합성 참조 문제", "synthetic"), ("CSV + 알려진 조건", "csv")],
        "synthetic",
    )
    data_contract = w.Text(description="데이터 JSON", layout=width, style=style)
    profile = dropdown(
        "실행 규모", [("본 실험", "research"), ("축소 동작 검사", "smoke")], "research"
    )
    execution = dropdown(
        "시작 방식", [("새 독립 실험", "new"), ("저장 실행에서 재개", "load")], "new"
    )
    batch = w.Checkbox(
        description="여러 방정식·민감도 조건을 한 번에 실행", value=False
    )
    batch_equations = OrderedEquationChecks(
        "방정식 묶음",
        [(v["label"], k) for k, v in EQUATIONS.items() if k != "cylinder_ns"],
        ("wave2d",),
    )
    directions = dropdown(
        "문제 방향",
        [
            ("순방향만", (False,)),
            ("역문제만", (True,)),
            ("순·역문제 모두", (False, True)),
        ],
        (False, True),
    )
    noise_levels = w.Text(
        value="0.0", description="노이즈 비율 목록", layout=width, style=style
    )
    initial_levels = w.Text(
        value="1.0", description="초기 계수 배율 목록", layout=width, style=style
    )
    load_settings = w.Button(description="저장 설정·모델 목록 불러오기")
    authorized = w.Checkbox(description="학습 실행 허용", value=False)
    settings_controls = settings_editor()
    advanced = w.Accordion(children=[settings_controls])
    advanced.set_title(0, "변수별 설명·학습 설정")
    advanced.selected_index = None
    trial = dropdown("완료 실험", [("모델 목록을 불러오세요", "")], "")
    coords = w.Textarea(
        value="[[0.25,0.25,0.5]]", description="좌표 JSON", layout=width, style=style
    )
    question = w.Textarea(
        value="wave2d의 Adam 결과를 비교해줘",
        description="질문",
        layout=width,
        style=style,
    )
    qwen = w.Checkbox(description="Qwen 사용", value=False)
    dense = w.Checkbox(description="E5 + BM25 검색", value=False)
    backend = dropdown("검색 엔진", [("NumPy", "numpy"), ("FAISS", "faiss")], "numpy")
    install = w.Button(description="FAISS 설치")
    install_note = w.HTML()
    time_value = w.FloatText(value=0, description="시간 위치", style=style)
    component = dropdown("표시할 출력", [("u", 0)], 0)
    extra_questions = w.Textarea(description="확인할 질문", layout=width, style=style)
    check_note = w.HTML(
        "Qwen은 기본 3문항, E5/FAISS는 검색 연결만 확인합니다. 질문을 입력하면 해당 질문만 사용합니다. 기존 전체 검사는 재실행하지 않습니다."
    )
    registry_action = dropdown(
        "모델 관리",
        [
            ("정보 조회", "info"),
            ("기본 추론 모델 지정", "select"),
            ("직전 모델 복원", "rollback"),
        ],
        "info",
    )
    approved = w.Checkbox(description="모델 변경 승인", value=False)
    run = w.Button(description="선택한 기능 실행", button_style="primary")
    out = w.Output()
    dependencies = w.Button(description="선택 기능의 필요 패키지 설치")
    dependency_note = w.HTML()
    fields = w.VBox()
    goal_note = w.HTML()
    future_steps = dropdown("외삽 평가 범위", [("1.01 ~ 3.00", 200), ("1.01 ~ 1.20", 20)], 200)
    future_note = w.HTML("저장한 순방향 기본 모델을 0.01 간격으로 평가합니다. CSV에는 전체 구간, 0.20 단위 구간, 마지막 시점의 오차가 기록됩니다.")
    training_choices = w.VBox([execution, generalization, data_mode, goal_note])
    state = {
        "root": None,
        "bundle": None,
        "service": None,
        "busy": False,
        "pending_training": None,
        "loaded_plan": (),
        "loaded_settings_path": None,
    }
    widgets = [
        action,
        storage,
        zip_name,
        equation,
        inverse,
        arms,
        seeds,
        generalization,
        data_mode,
        data_contract,
        profile,
        execution,
        authorized,
        trial,
        coords,
        question,
        qwen,
        dense,
        backend,
        time_value,
        registry_action,
        approved,
        batch,
        batch_equations,
        directions,
        noise_levels,
        initial_levels,
    ]

    def available_faiss():
        try:
            import faiss

            return faiss.IndexFlatIP(2).d == 2
        except (ImportError, AttributeError, OSError):
            return False

    def uses_equation_list():
        return batch.value and data_mode.value == "synthetic"

    def update_context(_=None):
        keys = batch_equations.value if uses_equation_list() else (equation.value,)
        settings_controls.set_context(
            arms.value,
            True in directions.value if batch.value else inverse.value,
            generalization.value,
            data_mode.value,
            any(k in EQUATIONS and EQUATIONS[k]["time_axis"] is not None for k in keys),
        )

    def update(_=None):
        training = action.value == "train"
        training_choices.layout.display = "" if training else "none"
        options = tuple(
            (spec["label"], key) for key, spec in EQUATIONS.items()
            if not training or (
                (generalization.value != "time_extrapolation" or spec["time_axis"] is not None)
                and (data_mode.value == "csv" or key != "cylinder_ns")
            )
        )
        if equation.options != options:
            previous = equation.value
            equation.options = options
            if any(key == previous for _, key in options):
                equation.value = previous
        selected = {
            "future": [future_steps, future_note],
            "predict": [trial, coords], "ask": [question, qwen, dense, backend],
            "plots": [trial, time_value, component],
            "registry": [equation, trial, registry_action, approved],
            "check": [qwen, dense, backend, extra_questions, check_note],
        }.get(action.value, [])
        batch_equations.set_available(k for _, k in options)
        if training:
            goal_note.value = (
                f"시간축이 있는 방정식 {len(options)}개를 선택할 수 있습니다."
                if generalization.value == "time_extrapolation" else ""
            )
            selected = []
            if data_mode.value == "csv":
                selected.append(data_contract)
            selected.extend([batch, batch_equations if uses_equation_list() else equation])
            selected.extend([directions if batch.value else inverse, arms, seeds, profile])
            if batch.value:
                if data_mode.value == "synthetic":
                    selected.append(noise_levels)
                if True in directions.value:
                    selected.append(initial_levels)
            selected.extend([advanced, authorized])
            locked = execution.value == "load"
            for control in (
                equation,
                inverse,
                arms,
                seeds,
                generalization,
                data_mode,
                profile,
                batch,
                batch_equations,
                directions,
                noise_levels,
                initial_levels,
                settings_controls,
            ):
                control.disabled = locked or state["busy"]
        elif not state["busy"]:
            equation.disabled = False
        update_context()
        missing = (
            action.value in {"ask", "check"}
            and dense.value
            and backend.value == "faiss"
            and not available_faiss()
        )
        install.layout.display = "" if missing else "none"
        install_note.value = (
            "FAISS 미설치: 아래 버튼을 직접 눌러 설치하세요." if missing else ""
        )
        fields.children = selected
        run.disabled = state["busy"] or missing
        run.description = (
            "학습 설정 확인·실행 준비"
            if action.value == "train"
            else "선택한 기능 실행"
        )
        time_value.layout.display = (
            "" if EQUATIONS[equation.value]["time_axis"] is not None else "none"
        )
        if state["root"] is not None and trial.value:
            selected_model()
        packages = missing_packages()
        dependencies.layout.display = "" if packages else "none"
        dependency_note.value = (
            "필요 패키지: " + ", ".join(packages) if packages else ""
        )

    def mount(_=None):
        with out:
            try:
                from google.colab import drive

                drive.mount("/content/drive")
            except Exception as exc:
                print(
                    "Drive 연결 실패:",
                    type(exc).__name__,
                    ". 로컬 ZIP 사용을 선택할 수 있습니다.",
                )

    def use_local(_=None):
        storage.value = "/content"
        state["root"] = None
        state["bundle"] = None
        state["service"] = None

    def missing_packages():
        import importlib.util

        required = (
            {"tensorflow": "tensorflow", "scipy": "scipy"}
            if action.value in {"predict", "plots", "future"}
            or (action.value == "train" and authorized.value)
            else {}
        )
        if action.value == "plots":
            required["matplotlib"] = "matplotlib"
        if action.value in {"ask", "check"} and qwen.value:
            required.update(torch="torch", transformers="transformers")
        if action.value in {"ask", "check"} and dense.value:
            required["sentence_transformers"] = "sentence-transformers"
        return [
            package
            for module, package in required.items()
            if importlib.util.find_spec(module) is None
        ]

    def dependencies_clicked(_):
        import subprocess, sys

        packages = missing_packages()
        if not packages or state["busy"]:
            return
        state["busy"] = True
        update()
        dependencies.disabled = True
        with out:
            try:
                result = subprocess.run(
                    [
                        sys.executable,
                        "-m",
                        "pip",
                        "install",
                        "--quiet",
                        "--no-input",
                        *packages,
                    ],
                    capture_output=True,
                    text=True,
                    timeout=600,
                )
                if result.returncode:
                    raise RuntimeError(result.stderr[-1200:])
                print(
                    "필요 패키지 설치 완료. 실행 버튼을 누르세요. 이미 불완전하게 import한 패키지가 있으면 런타임을 재시작하세요."
                )
            except Exception as exc:
                print("설치 실패:", str(exc))
            finally:
                state["busy"] = False
                dependencies.disabled = False
                update()

    def install_clicked(_):
        if state["busy"] or available_faiss():
            return
        import subprocess, sys

        state["busy"] = True
        update()
        install.disabled = True
        with out:
            try:
                r = subprocess.run(
                    [
                        sys.executable,
                        "-m",
                        "pip",
                        "install",
                        "--quiet",
                        "--no-input",
                        "--no-deps",
                        "--only-binary=:all:",
                        "faiss-cpu",
                    ],
                    capture_output=True,
                    text=True,
                    timeout=240,
                )
                if r.returncode or not available_faiss():
                    raise RuntimeError("설치 후 로드 실패")
                print("FAISS 설치 완료. 실행 버튼을 누르세요.")
            except Exception as exc:
                print(
                    "설치 실패:",
                    type(exc).__name__,
                    ". NumPy 검색 또는 런타임 재시작을 사용할 수 있습니다.",
                )
            finally:
                state["busy"] = False
                install.disabled = False
                update()

    def open_root():
        if Path(
            zip_name.value
        ).name != zip_name.value or not zip_name.value.lower().endswith(".zip"):
            raise ValueError("ZIP에는 파일명만 입력하세요.")
        file = Path(storage.value) / zip_name.value
        if state["root"] is None or state["bundle"] != str(file):
            if not file.is_file():
                raise FileNotFoundError("선택 폴더에 " + file.name + "을 넣으세요.")
            workspace = Path(tempfile.mkdtemp(prefix="v4_view_")) / "results"
            restore(file, workspace)
            state.update(root=workspace, bundle=str(file), service=None)
        return state["root"]

    def current_service(root):
        service = state["service"]
        if service is None or service.root != root:
            service = state["service"] = Service(root)
        return service

    def refresh(_=None):
        state["loaded_settings_path"] = None
        state["pending_training"] = None
        authorized.value = False
        with out:
            try:
                root = open_root()
                kind = json.loads((root / "contract.json").read_text(encoding="utf-8"))[
                    "kind"
                ]
                if kind not in {"results", "catalog"}:
                    raise ValueError(
                        "결과 ZIP을 선택하세요. 재개 ZIP은 같은 실행 ID의 결과 ZIP과 함께 자동으로 사용합니다."
                    )
                service = current_service(root)
                options = [
                    (
                        EQUATIONS[m["equation"]]["label"]
                        + " / "
                        + ("역" if m["inverse"] else "순")
                        + " / "
                        + ARM_LABELS[m["arm"]]
                        + " / "
                        + str(m["seed"])
                        + " / "
                        + m["generalization"]["mode"],
                        m["trial_id"],
                    )
                    for m in service.models()
                ]
                trial.options = options or [("완료된 V4 모델 없음", "")]
                if (root / "settings.json").exists():
                    values = json.loads(
                        (root / "settings.json").read_text(encoding="utf-8")
                    )
                    settings = Settings.from_dict(values)
                    settings_controls.value = values
                    generalization.value = settings.generalization
                    data_mode.value = settings.data_mode
                    equation.value = settings.equation
                    inverse.value = settings.inverse
                    arms.value = settings.arms
                    if settings.seeds in ((3234,), SEEDS):
                        seeds.value = settings.seeds
                    authorized.value = False
                    profile.value = (
                        "smoke" if settings.cap < settings.stop_first else "research"
                    )
                execution.value = "load" if kind == "results" else "new"
                plan_path = root / "plan.json"
                plan = (
                    tuple(
                        Settings.from_dict(s)
                        for s in json.loads(plan_path.read_text(encoding="utf-8"))
                    )
                    if kind == "results" and plan_path.is_file()
                    else ()
                )
                state["loaded_plan"] = plan
                batch.value = bool(plan)
                if plan:
                    batch_equations.value = tuple(
                        dict.fromkeys(s.equation for s in plan)
                    )
                    directions.value = tuple(sorted({s.inverse for s in plan}))
                    noise_levels.value = ", ".join(
                        str(x) for x in dict.fromkeys(s.noise_fraction for s in plan)
                    )
                    if any(s.inverse for s in plan):
                        multipliers = dict.fromkeys(
                            s.initial_coefficient
                            / (
                                settings.initial_coefficient
                                or EQUATIONS[s.equation]["initial"]
                            )
                            for s in plan
                            if s.inverse and s.initial_coefficient is not None
                        )
                        initial_levels.value = (
                            ", ".join(f"{x:g}" for x in multipliers) or "1.0"
                        )
                state["loaded_settings_path"] = (
                    state["bundle"] if kind == "results" else None
                )
                update()
                print(
                    "저장 설정과 모델 목록을 불러왔습니다. 시작 방식: "
                    + (
                        "저장 실행에서 재개"
                        if kind == "results"
                        else "새 독립 실험 · 결과 목록 ZIP"
                    )
                    + ". 학습 허용은 꺼져 있습니다."
                )
            except Exception as exc:
                print("불러오기 실패:", str(exc))

    def selected_model(_=None):
        if state["root"] is None or not trial.value:
            return
        info = current_service(state["root"]).model_info(trial.value)
        component.options = [
            (name, i) for i, name in enumerate(info["output_variables"])
        ]
        time_axis = EQUATIONS[info["equation"]]["time_axis"]
        time_value.layout.display = "" if time_axis is not None else "none"
        if time_axis is not None:
            lo, hi = info["input_variables"][time_axis]["range"]
            if not lo <= time_value.value <= hi:
                time_value.value = (lo + hi) / 2

    def training_signature():
        return digest(
            {
                "settings": settings_controls.value,
                "equation": equation.value,
                "inverse": inverse.value,
                "arms": list(arms.value),
                "seeds": list(seeds.value),
                "generalization": generalization.value,
                "data_mode": data_mode.value,
                "profile": profile.value,
                "execution": execution.value,
                "storage": storage.value,
                "zip": zip_name.value,
                "data_contract": data_contract.value,
                "batch": batch.value,
                "batch_equations": list(batch_equations.value),
                "directions": directions.value,
                "noise_levels": noise_levels.value,
                "initial_levels": initial_levels.value,
            }
        )

    def execute(_=None, *, from_cell=False):
        if state["busy"]:
            return
        state["busy"] = True
        for widget in widgets:
            widget.disabled = True
        settings_controls.disabled = True
        connect.disabled = local.disabled = install.disabled = True
        update()
        with nullcontext() if from_cell else out:
            if not from_cell:
                out.clear_output(wait=True)
            try:
                selected = action.value
                if selected == "train":
                    equations = batch_equations.value if uses_equation_list() else (equation.value,)
                    if not equations:
                        raise ValueError("방정식 묶음에서 하나 이상 선택하세요.")
                    values = settings_controls.value
                    values.update(
                        equation=equation.value if execution.value == "load" else equations[0],
                        inverse=inverse.value,
                        arms=list(arms.value),
                        seeds=list(seeds.value),
                        generalization=generalization.value,
                        data_mode=data_mode.value,
                    )
                    settings = Settings.from_dict(values)
                    if profile.value == "smoke":
                        settings = settings.smoke()
                    if execution.value == "load" and state[
                        "loaded_settings_path"
                    ] != str(Path(storage.value) / zip_name.value):
                        raise ValueError(
                            "선택한 ZIP의 저장 설정·모델 목록을 먼저 불러오세요."
                        )
                    configurations = (
                        state["loaded_plan"]
                        if execution.value == "load" and state["loaded_plan"]
                        else (
                            experiment_plan(
                                settings,
                                equations,
                                directions.value,
                                (0.0,) if data_mode.value == "csv" else tuple(
                                    float(v.strip())
                                    for v in noise_levels.value.split(",")
                                ),
                                (1.0,) if True not in directions.value else tuple(
                                    float(v.strip())
                                    for v in initial_levels.value.split(",")
                                ),
                            )
                            if batch.value
                            else [settings]
                        )
                    )
                    print(
                        "목적: 독립 V4 "
                        + (
                            "축소 동작 검사"
                            if profile.value == "smoke"
                            else "학습 비교"
                        )
                    )
                    trial_count = sum(
                        len(s.arms) * len(s.seeds) for s in configurations
                    )
                    print(
                        "실행 범위:",
                        trial_count,
                        "실험 / 실험별 Adam 한도",
                        settings.cap,
                        "/ L-BFGS",
                        settings.lbfgs_maxiter,
                    )
                    print(
                        "Adam 최악 한도:",
                        trial_count * settings.cap,
                        "업데이트. 완료된 실험은 제외합니다.",
                    )
                    descriptions = {
                        "baseline": "관측 데이터·물리식·경계 및 초기조건을 함께 학습",
                        "loss_sa": "세 손실 그룹의 가중치를 학습 중 조절",
                        "ff": "입력을 주파수 특징으로 변환해 학습",
                        "ff_loss_sa": "주파수 특징과 손실 가중치 조절을 함께 사용",
                        "ff_curriculum": "주파수 특징을 사용하며 PDE 비중을 점진적으로 증가",
                        "supervised_only": "물리식 없이 관측값만 학습; 계수 추정 없음",
                    }
                    display(
                        Markdown(
                            "| 선택 방법 | 학습 방식 |\n|---|---|\n"
                            + "\n".join(
                                f"| {ARM_LABELS[k]} | {descriptions[k]} |"
                                for k in settings.arms
                            )
                            + "\n\n"
                            + ("방정식 실행 순서: **" if batch.value else "방정식: **")
                            + " → ".join(
                                dict.fromkeys(
                                    EQUATIONS[s.equation]["label"]
                                    for s in configurations
                                )
                            )
                            + "** · "
                            + "·".join(
                                dict.fromkeys(
                                    "역문제" if s.inverse else "순방향"
                                    for s in configurations
                                )
                            )
                            + f" · 시드 {list(settings.seeds)}\n\n"
                            f"목표: 검증 **L1·L2 모두 {settings.target * 100:g}% 이하** · "
                            f"{settings.stop_first:,}회부터 {settings.check_every:,}회마다 확인, "
                            f"연속 {settings.consecutive}회 만족 시 중단.\n\n"
                            f"학습점 갱신: {settings.sampling_every:,}회마다 "
                            "PDE는 RAD, 나머지 합성 학습점은 LHS. "
                            "검증·평가점은 고정합니다. 상세 수치는 접힌 고급 설정에서 확인합니다."
                        )
                    )
                    if not authorized.value:
                        state["pending_training"] = None
                        print(
                            "설정 확인만 완료했습니다. 학습 실행 허용을 켜야 실행합니다."
                        )
                        return
                    if missing_packages():
                        raise RuntimeError("먼저 필요 패키지 설치 버튼을 누르세요.")
                    if not from_cell:
                        state["pending_training"] = training_signature()
                        print(
                            "실행 준비 완료. 아래 ‘학습 실행’ 셀의 ▶를 누르세요. 이 버튼에서는 학습하지 않습니다."
                        )
                        return
                    target = Path(storage.value)
                    if (
                        str(target).startswith("/content/drive/")
                        and not Path("/content/drive/MyDrive").is_dir()
                    ):
                        mount()
                        if not Path("/content/drive/MyDrive").is_dir():
                            raise RuntimeError(
                                "Drive 연결 또는 로컬 ZIP 사용이 필요합니다."
                            )
                    target.mkdir(parents=True, exist_ok=True)
                    if execution.value == "load":
                        root = open_root()
                        stored = Settings.from_dict(
                            json.loads(
                                (root / "settings.json").read_text(encoding="utf-8")
                            )
                        )
                        if (
                            json.loads(
                                (root / "contract.json").read_text(encoding="utf-8")
                            )["kind"]
                            != "results"
                        ):
                            raise ValueError(
                                "결과 목록 ZIP에서는 새 독립 실험을 선택하세요."
                            )
                        if stored.to_dict() != settings.to_dict():
                            raise ValueError(
                                "재개 시 저장 설정·모델 목록을 불러오고 값을 그대로 사용하세요. 설정 변경은 새 실험입니다."
                            )
                        source = DataSource(
                            settings,
                            (
                                root / "inputs/data.json"
                                if settings.data_mode == "csv"
                                else None
                            ),
                        )
                        if (root / "plan.json").is_file():
                            configurations = [
                                Settings.from_dict(v)
                                for v in json.loads(
                                    (root / "plan.json").read_text(encoding="utf-8")
                                )
                            ]
                    else:
                        root = Path(tempfile.mkdtemp(prefix="v4_run_")) / "run"
                        source = create_run(
                            root,
                            settings,
                            (
                                (target / data_contract.value)
                                if settings.data_mode == "csv"
                                else None
                            ),
                        )
                    owner = json.loads(
                        (root / "contract.json").read_text(encoding="utf-8")
                    )
                    suffix = owner["run_id"]
                    result = (
                        (target / zip_name.value)
                        if execution.value == "load"
                        else target / ("V4_Results_" + suffix + ".zip")
                    )
                    resume = target / ("V4_Resume_" + suffix + ".zip")
                    pack(root, result)
                    print("결과 ZIP:", result.name, "/ 재개 ZIP:", resume.name)
                    if execution.value == "load" and not resume.exists():
                        print(
                            "재개 ZIP 없음: 완료 실험은 건너뛰며 저장된 미완료 상태가 없는 실험은 처음부터 시작합니다."
                        )
                    from .progress import training_progress

                    with training_progress():
                        if len(configurations) > 1 or batch.value:
                            execute_plan(root, configurations, result, resume, True)
                        else:
                            run_all(root, settings, source, result, resume, True)
                    trial.options = [("모델 목록을 불러오세요", "")]
                    state.update(root=root, bundle=str(result), service=None)
                    zip_name.value = result.name
                    print("완료:", result.name)
                    return
                root = open_root()
                service = current_service(root)
                if selected == "results":
                    file = root / "REPORT.md"
                    display(
                        Markdown(
                            file.read_text(encoding="utf-8")
                            if file.exists()
                            else "완료된 V4 실험이 없습니다."
                        )
                    )
                    historical = root / "historical/V3_comparison.md"
                    if historical.exists():
                        display(Markdown(historical.read_text(encoding="utf-8")))
                elif selected == "future":
                    from .future_evaluation import evaluate_saved, save_evaluation

                    steps = future_steps.value
                    rows = evaluate_saved(root, steps=steps)
                    output = Path(storage.value) / (f"V4_Future{steps}_" + uuid.uuid4().hex[:8] + ".zip")
                    save_evaluation(rows, output, steps=steps)
                    lines = ["| 방정식 | 방법 | 시드 | 단계 | 미래 L1 (%) | 미래 L2 (%) |",
                             "|---|---|---:|---|---:|---:|"]
                    for row in rows:
                        lines.append(
                            f"| {EQUATIONS[row['equation']]['label']} | {ARM_LABELS[row['arm']]} | "
                            f"{row['seed']} | {row['stage']} | {row['future_l1_pct']:.4f} | "
                            f"{row['future_l2_pct']:.4f} |"
                        )
                    display(Markdown("\n".join(lines)))
                    print("평가 결과:", output.name)
                elif selected == "predict":
                    print(service.predict(trial.value, json.loads(coords.value)))
                elif selected == "ask":
                    result = service.ask(
                        question.value, qwen.value, dense.value, backend.value
                    )
                    print(result["answer"])
                    print("Qwen 실제 호출:", result["llm_called"])
                elif selected == "registry":
                    if registry_action.value == "info":
                        info = service.model_info(trial.value)
                        lines = [
                            "### 모델 정보",
                            "",
                            "| 항목 | 값 |",
                            "|---|---|",
                            f"| 방정식 | {EQUATIONS[info['equation']]['label']} |",
                            f"| 방법 | {ARM_LABELS[info['arm']]} |",
                            f"| 시드 | {info['seed']} |",
                            f"| 선택 단계 | {info['selected_stage']} |",
                            "",
                            "| 입력 변수 | 범위 |",
                            "|---|---|",
                        ]
                        lines += [
                            f"| {v['symbol']} | {v['range'][0]} ~ {v['range'][1]} |"
                            for v in info["input_variables"]
                        ]
                        lines += [
                            "",
                            "출력: " + ", ".join(info["output_variables"]),
                            "추정 대상: " + info["coefficient"],
                        ]
                        display(Markdown("\n".join(lines)))
                    else:
                        result = (
                            service.register(
                                equation.value, trial.value, approved.value
                            )
                            if registry_action.value == "select"
                            else service.rollback(equation.value, approved.value)
                        )
                        print(result)
                        output = Path(storage.value) / (
                            "V4_Registry_" + uuid.uuid4().hex[:8] + ".zip"
                        )
                        pack(root, output)
                        print("출력:", output.name)
                elif selected == "check":
                    from .verification import integration_checks
                    from .storage import write_json

                    checks = integration_checks(
                        root,
                        qwen.value,
                        dense.value,
                        backend.value,
                        extra_questions.value.splitlines(),
                        service=service,
                    )
                    write_json(root / "validation/colab.json", checks)
                    output = Path(storage.value) / (
                        "V4_Check_" + uuid.uuid4().hex[:8] + ".zip"
                    )
                    pack(root, output)
                    print(
                        "선택 기능 확인: 질문",
                        checks["question_count"],
                        "개 / 실제 Qwen 호출",
                        checks["actual_llm_calls"],
                    )
                    if dense.value:
                        print(
                            "검색 연결:",
                            checks["retrieval"]["backend"],
                            checks["retrieval"]["passed"],
                        )
                    print(
                        "출력:",
                        output.name,
                        "/ 답변 의미 정확도는 별도 확인 항목입니다.",
                    )
                else:
                    if selected == "documents":
                        generate_report(root)
                    else:
                        for file in make_plots(
                            root, trial.value or None, time_value.value, component.value
                        ):
                            if file.suffix == ".png":
                                from IPython.display import Image

                                display(Image(filename=str(file)))
                    output = Path(storage.value) / (
                        "V4_"
                        + ("Docs" if selected == "documents" else "Plots")
                        + "_"
                        + uuid.uuid4().hex[:8]
                        + ".zip"
                    )
                    pack(root, output)
                    print("출력:", output.name)
            except Exception as exc:
                print("실행하지 못했습니다:", type(exc).__name__, str(exc))
                if from_cell:
                    raise
            finally:
                state["busy"] = False
                for widget in widgets:
                    widget.disabled = False
                settings_controls.disabled = False
                connect.disabled = local.disabled = install.disabled = False
                update()

    def run_training():
        """Called by a regular notebook cell, never by the widget callback."""
        if state["busy"]:
            raise RuntimeError("다른 기능이 실행 중입니다.")
        if action.value != "train" or not authorized.value:
            state["pending_training"] = None
            raise PermissionError(
                "학습 설정·실행을 선택하고 학습 실행 허용을 켠 뒤 실행 준비를 누르세요."
            )
        pending = state["pending_training"]
        if pending is None:
            raise RuntimeError("먼저 패널의 학습 설정 확인·실행 준비 버튼을 누르세요.")
        if pending != training_signature():
            state["pending_training"] = None
            raise RuntimeError(
                "준비 이후 설정이 변경됐습니다. 설정 확인·실행 준비를 다시 누르세요."
            )
        state["pending_training"] = None
        return execute(from_cell=True)

    for widget in (
        action,
        equation,
        data_mode,
        dense,
        backend,
        qwen,
        authorized,
        batch,
        execution,
        directions,
        inverse,
        generalization,
    ):
        widget.observe(update, names="value")
    for checkbox in arms.checks.values():
        checkbox.observe(update, names="value")
    for checkbox in batch_equations.checks.values():
        checkbox.observe(update_context, names="value")
    run.on_click(execute)
    connect.on_click(mount)
    local.on_click(use_local)
    install.on_click(install_clicked)
    dependencies.on_click(dependencies_clicked)
    load_settings.on_click(refresh)
    update()
    trial.observe(selected_model, names="value")
    dashboard = w.VBox(
        [
            w.HTML("<h3>설정</h3>"),
            action,
            training_choices,
            storage,
            zip_name,
            w.HBox([connect, local, load_settings]),
            fields,
            dependency_note,
            dependencies,
            install_note,
            install,
            run,
            w.HTML("<h3>결과 · 실행 로그</h3>"),
            out,
        ]
    )
    if show:
        display(dashboard)
    return {
        "dashboard": dashboard,
        "state": state,
        "execute": execute,
        "run_training": run_training,
        "load_settings": refresh,
        "run_button": run,
        "output": out,
        "widgets": {
            "action": action,
            "equation": equation,
            "arms": arms,
            "batch": batch,
            "generalization": generalization,
            "data_mode": data_mode,
            "directions": directions,
            "batch_equations": batch_equations,
            "authorized": authorized,
            "settings": settings_controls,
            "execution": execution,
            "zip": zip_name,
            "storage": storage,
            "qwen": qwen,
            "dense": dense,
            "backend": backend,
            "questions": extra_questions,
            "question": question,
        },
    }


In [ ]:
%%writefile "{V4_MODULE_DIR}/metrics.py"
"""Field, coefficient, local residual and integrated-equation balance reports."""

import numpy as np
from .generalization import field_metrics


def observed_metrics(reference, prediction):
    reference = np.asarray(reference)
    prediction = np.asarray(prediction)
    components = []
    for a in range(reference.shape[1]):
        valid = np.isfinite(reference[:, a])
        components.append(
            field_metrics(reference[valid, a], prediction[valid, a])
            if valid.any()
            else None
        )
    defined = [c for c in components if c is not None and c["relative_errors_defined"]]
    all_observed_defined = len(defined) == sum(c is not None for c in components)
    return {
        "relative_l1": (
            max((c["relative_l1"] for c in defined), default=None)
            if all_observed_defined
            else None
        ),
        "relative_l2": (
            max((c["relative_l2"] for c in defined), default=None)
            if all_observed_defined
            else None
        ),
        "all_observed_relative_errors_defined": all_observed_defined,
        "components": components,
        "observed_components": len(defined),
        "aggregation": "maximum per observed output; unobserved outputs are not scored",
    }


def constraint_metrics(runtime, arrays):
    results = {}
    scale = np.asarray(runtime.scales["field"])
    for family in ("boundary", "initial", "gauge"):
        if family not in arrays:
            continue
        prediction = runtime.predict(arrays[family])
        labels = arrays["label_" + family]
        observed = np.isfinite(labels)
        if family == "boundary" and len(scale) == 3:
            observed[:, 2] = False
        if family == "gauge":
            observed[:, :2] = False
        error = (prediction - labels) / scale
        results[family] = {
            "normalized_rms": (
                float(np.sqrt(np.mean(error[observed] ** 2)))
                if observed.any()
                else None
            ),
            "normalized_max": (
                float(np.max(abs(error[observed]))) if observed.any() else None
            ),
            "points": len(labels),
            "source": "fixed known constraints; CSV conditions may also be training constraints",
        }
    if "initial_velocity" in arrays:
        points = arrays.get("initial_velocity_x", arrays["initial"])
        _, g, _ = runtime.derivatives(points)
        duration = np.ptp(np.asarray(runtime.source.domain)[-1])
        error = (
            (g.numpy()[:, 0, -1:] - arrays["initial_velocity"]) * duration / scale[0]
        )
        results["initial_velocity"] = {
            "normalized_rms": float(np.sqrt(np.mean(error**2))),
            "normalized_max": float(abs(error).max()),
        }
    return results


def evaluate(runtime, data, physics_points, iteration, stage, constraints=None):
    before = runtime.snapshot()
    field = (
        observed_metrics(data["y"], runtime.predict(data["x"]))
        if data is not None
        else None
    )
    parts = []
    for i in range(0, len(physics_points), 128):
        parts.append(runtime.pde_residual(physics_points[i : i + 128]).numpy())
    residuals = np.concatenate(parts)
    scaled = residuals / np.asarray(runtime.scales["residual"])
    volume = float(np.prod(np.ptp(np.asarray(runtime.source.domain), axis=1)))
    truth = runtime.source.truth
    coefficient = (
        float(runtime.coefficient) if runtime.arm != "supervised_only" else None
    )
    constraint_result = constraint_metrics(runtime, constraints or {})
    from .balance import balance_report

    balance = balance_report(runtime)
    true_view = None
    if runtime.inverse and truth is not None:
        parts = [
            runtime.pde_residual(physics_points[i : i + 128], truth).numpy()
            for i in range(0, len(physics_points), 128)
        ]
        true_view = np.sqrt(
            np.mean(
                (np.concatenate(parts) / np.asarray(runtime.scales["residual"])) ** 2,
                axis=0,
            )
        ).tolist()
    runtime.assert_unchanged(before)
    return {
        "stage": stage,
        "iteration": iteration,
        "field": field,
        "coefficient": coefficient,
        "coefficient_relative_error": (
            abs(coefficient / truth - 1)
            if truth is not None and coefficient is not None and runtime.inverse
            else None
        ),
        "coefficient_estimated": runtime.inverse,
        "constraints": constraint_result,
        "integral_balance": balance,
        "physics": {
            "point_count": len(physics_points),
            "normalized_rms": np.sqrt(np.mean(scaled**2, axis=0)).tolist(),
            "normalized_p99": np.quantile(abs(scaled), 0.99, axis=0).tolist(),
            "normalized_max": abs(scaled).max(axis=0).tolist(),
            "integrated_equation_balance_defect": (
                volume * np.mean(residuals, axis=0)
            ).tolist(),
            "balance_definition": "quadrature estimate of signed governing-equation residual; not an independent surface-flux certificate",
            "reference_coefficient_normalized_rms": true_view,
            "normalization": runtime.scales,
            "used_for_stopping": False,
        },
        "state_guard": "PASS",
    }


In [ ]:
%%writefile "{V4_MODULE_DIR}/progress.py"
"""Scoped console instrumentation; no additional model or gradient evaluation."""

from contextlib import contextmanager, ExitStack
from functools import wraps
import threading
import time
from pathlib import Path
import shutil
from unittest.mock import patch

_LOCK = threading.Lock()


@contextmanager
def training_progress(log=print, interval_seconds=15.0):
    """Instrument synchronous Colab training and restore hooks even on interruption.

    Hooks are process-wide while active, so concurrent instrumented runs are
    rejected. Numerical runner/runtime source and checkpoint identity stay intact.
    """
    if not _LOCK.acquire(blocking=False):
        raise RuntimeError("다른 학습의 진행 표시가 실행 중입니다.")
    try:
        from scipy import optimize
        from . import runner
        from .runtime import Runtime
        from .diagnostics import observe
        from . import diagnostics
        from .storage import write_json, sha_file

        original_lbfgs = Runtime.lbfgs
        original_evaluate = runner.evaluate
        original_pack = runner.pack
        original_resume = runner.resume_slot
        original_trial = runner.run_trial
        resume_depth = 0

        def emit(message):
            log(message)

        def label(runtime):
            direction = "역문제" if runtime.settings.inverse else "순방향"
            return f"{runtime.settings.equation} [{direction}] {runtime.arm}"

        @wraps(original_trial)
        def trial(root, settings, arm, seed, *args, **kwargs):
            direction = "역문제" if settings.inverse else "순방향"
            emit(f"실험 확인: {settings.equation} · {direction} · {arm} · 시드 {seed}")
            # Diagnostic provenance is separate from immutable training identity.
            source = Path(diagnostics.__file__)
            checksum = sha_file(source)
            destination = Path(root) / "validation" / "observer"
            destination.mkdir(parents=True, exist_ok=True)
            stem = f"{diagnostics.REVISION}_{checksum[:12]}"
            shutil.copyfile(source, destination / (stem + ".py"))
            write_json(
                destination / (stem + ".json"),
                {
                    "revision": diagnostics.REVISION,
                    "diagnostic_source_sha256": checksum,
                    "training_source_sha": runner.training_source_digest(),
                    "training_protocol_changed": False,
                    "historical_diagnostics_rewritten": False,
                },
            )
            return original_trial(root, settings, arm, seed, *args, **kwargs)

        @wraps(original_lbfgs)
        def lbfgs(runtime, tensors, completed):
            trial_label = label(runtime)
            started = time.perf_counter()
            emit(f"{trial_label} L-BFGS 시작 · 최대 {runtime.settings.lbfgs_maxiter}회")
            original_minimize = optimize.minimize

            @wraps(original_minimize)
            def minimize(fun, *args, **kwargs):
                if kwargs.get("method") != "L-BFGS-B":
                    return original_minimize(fun, *args, **kwargs)
                calls = 0
                accepted = 0
                last_value = None
                last_message = started
                callback = kwargs.get("callback")

                def report(force=False):
                    nonlocal last_message
                    now = time.perf_counter()
                    if force or now - last_message >= interval_seconds:
                        value = "—" if last_value is None else f"{last_value:.6g}"
                        emit(
                            f"{trial_label} L-BFGS 진행 · 반복 {accepted}/{runtime.settings.lbfgs_maxiter}"
                            f" · 손실·기울기 평가 {calls}회 · 최근 목적함수 {value}"
                            f" · 경과 {now - started:.0f}초"
                        )
                        last_message = now

                @wraps(fun)
                def objective(*position_args, **position_kwargs):
                    nonlocal calls, last_value
                    result = fun(*position_args, **position_kwargs)
                    calls += 1
                    last_value = float(
                        result[0] if isinstance(result, tuple) else result
                    )
                    report(force=calls == 1)
                    return result

                def accepted_step(xk):
                    nonlocal accepted
                    accepted += 1
                    report()
                    if callback is not None:
                        return callback(xk)

                kwargs["callback"] = accepted_step
                result = original_minimize(objective, *args, **kwargs)
                report(force=True)
                return result

            with patch.object(optimize, "minimize", minimize):
                result = original_lbfgs(runtime, tensors, completed)
            emit(
                f"{trial_label} L-BFGS 종료 · 반복 {result['iterations']}회"
                f" · 평가 {result['function_evaluations']}회"
                f" · {time.perf_counter() - started:.1f}초 · {result['message']}"
            )
            return result

        @wraps(original_evaluate)
        def evaluate(runtime, evaluation, physics, iteration, stage, *args, **kwargs):
            trial_label = label(runtime)
            if stage == "adam":
                emit(f"{trial_label} Adam 학습 종료 · {iteration}회 · 최종 평가 시작")
                if not runtime.settings.lbfgs_maxiter:
                    emit(f"{trial_label} L-BFGS 생략 · 설정 0회")
            else:
                emit(f"{trial_label} {stage} 최종 평가 시작")
            started = time.perf_counter()
            result = original_evaluate(
                runtime, evaluation, physics, iteration, stage, *args, **kwargs
            )
            emit(
                f"{trial_label} {stage} 최종 평가 완료 · {time.perf_counter() - started:.1f}초"
            )
            return result

        def save_with_notice(original, label, *args, **kwargs):
            started = time.perf_counter()
            emit(f"{label} 저장·무결성 검사 시작")
            result = original(*args, **kwargs)
            emit(f"{label} 저장 완료 · {time.perf_counter() - started:.1f}초")
            return result

        @wraps(original_pack)
        def pack(*args, **kwargs):
            if resume_depth:
                return original_pack(*args, **kwargs)
            return save_with_notice(original_pack, "결과 ZIP", *args, **kwargs)

        @wraps(original_resume)
        def resume(*args, **kwargs):
            nonlocal resume_depth
            resume_depth += 1
            try:
                return save_with_notice(original_resume, "재개 ZIP", *args, **kwargs)
            finally:
                resume_depth -= 1

        with ExitStack() as stack:
            stack.enter_context(patch.object(Runtime, "observe", observe))
            stack.enter_context(patch.object(runner, "run_trial", trial))
            stack.enter_context(patch.object(Runtime, "lbfgs", lbfgs))
            stack.enter_context(patch.object(runner, "evaluate", evaluate))
            stack.enter_context(patch.object(runner, "pack", pack))
            stack.enter_context(patch.object(runner, "resume_slot", resume))
            yield
    finally:
        _LOCK.release()


In [ ]:
%%writefile "{V4_MODULE_DIR}/reporting.py"
"""Reports distinguish Adam, candidate L-BFGS and selected inference stages."""

import csv
import json
import re
from pathlib import Path
import numpy as np
from .config import ARM_LABELS
from .equations import EQUATIONS
from .storage import write_json


def reports(root):
    records = []
    for p in sorted((Path(root) / "trials").glob("*/report.json")):
        record = json.loads(p.read_text(encoding="utf-8"))
        if record.get("trial_id") != p.parent.name or not re.fullmatch(
            "[0-9a-f]{16}", p.parent.name
        ):
            raise ValueError("완료 실험 ID 또는 보고서 경로 오류")
        records.append(record)
    return records


def comparison(root):
    rows = []
    for r in reports(root):
        for stage in ("adam", "lbfgs"):
            result = r.get(stage)
            if result is None:
                continue
            field = result["field"] or {}
            rows.append(
                {
                    "trial_id": r["trial_id"],
                    "equation": r["equation"],
                    "inverse": r["inverse"],
                    "generalization": r["generalization"]["mode"],
                    "arm": r["arm"],
                    "seed": r["seed"],
                    "stage": stage,
                    "l1": field.get("relative_l1"),
                    "l2": field.get("relative_l2"),
                    "coefficient_error": result["coefficient_relative_error"],
                    "adam_updates": r["adam"]["optimizer_updates"],
                    "field_target": r["field_target"],
                    "execution_scope": r["execution_scope"],
                    "protocol_sha": r["protocol_sha"],
                    "noise_fraction": r["condition"]["noise_fraction"],
                    "initial_coefficient": r["condition"]["initial_coefficient"],
                    "lbfgs_iterations": result.get("solver", {}).get("iterations"),
                    "lbfgs_evaluations": result.get("solver", {}).get(
                        "function_evaluations"
                    ),
                }
            )
    return rows


def generate_report(root):
    root = Path(root)
    rows = comparison(root)
    grouped = {}
    for row in rows:
        key = (
            row["equation"],
            row["inverse"],
            row["generalization"],
            row["arm"],
            row["stage"],
            row["execution_scope"],
            row["noise_fraction"],
            row["initial_coefficient"],
            row["protocol_sha"],
        )
        grouped.setdefault(key, []).append(row)
    aggregates = []
    text = [
        "# 온라인 PINN PDE 프레임워크 V4",
        "",
        "Adam과 L-BFGS 종료를 별도로 비교합니다. 아래 수치는 저장된 최종 평가 결과입니다.",
        "",
        "| 방정식 | 문제 | 분할 | 방법 | 단계 | 조건 (노이즈/초기계수) | 시드 | L1 평균 % | L2 평균 % | L2 표본 표준편차 % |",
        "|---|---|---|---|---|---|---:|---:|---:|---:|",
    ]
    for key, group in grouped.items():
        eq, inv, mode, arm, stage, scope, noise, initial, protocol = key
        l1 = [r["l1"] for r in group if r["l1"] is not None]
        l2 = [r["l2"] for r in group if r["l2"] is not None]
        mean1 = float(np.mean(l1)) if l1 else None
        mean2 = float(np.mean(l2)) if l2 else None
        std = float(np.std(l2, ddof=1)) if len(l2) > 1 else None
        aggregates.append(
            {
                "equation": eq,
                "inverse": inv,
                "mode": mode,
                "arm": arm,
                "stage": stage,
                "execution_scope": scope,
                "noise_fraction": noise,
                "initial_coefficient": initial,
                "protocol_sha": protocol,
                "seeds": len(group),
                "mean_l1": mean1,
                "mean_l2": mean2,
                "sample_std_l2": std,
                "field_target_count": sum(
                    r["l1"] is not None
                    and r["l2"] is not None
                    and max(r["l1"], r["l2"]) <= r["field_target"]
                    for r in group
                ),
            }
        )
        fmt = lambda v: "해당 없음" if v is None else f"{100*v:.5f}"
        text.append(
            f"| {EQUATIONS[eq]['label']} | {'역문제' if inv else '순방향'} | {mode} / {scope} | {ARM_LABELS[arm]} | {stage} | {noise}/{initial} ({protocol[:8]}) | {len(group)} | {fmt(mean1)} | {fmt(mean2)} | {fmt(std)} |"
        )
    if not rows:
        text += ["", "아직 완료된 V4 학습 실험이 없습니다."]
    text += [
        "",
        "계수 오차·PDE/경계/초기조건·진단·iter·후처리 선택 근거는 개별 실험 JSON에 있습니다.",
        "보간 시 시드별 제외 구역이 다르면 표본 변동에는 구역 난이도 차이가 포함됩니다.",
        "Supervised-only는 계수를 추정하지 않습니다. 실제 데이터에서 없는 정답은 평가값으로 생성하지 않습니다.",
    ]
    (root / "REPORT.md").write_text("\n".join(text) + "\n", encoding="utf-8")
    if rows:
        with (root / "comparison.csv").open(
            "w", encoding="utf-8-sig", newline=""
        ) as stream:
            writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
            writer.writeheader()
            writer.writerows(rows)
    write_json(
        root / "summary.json",
        {
            "format": "v4-independent-1",
            "completed_trials": len(reports(root)),
            "rows": rows,
            "aggregates": aggregates,
            "historical_v3_not_v4_results": True,
        },
    )
    catalog = "# V4 기능 목록\n\n- 12개 방정식 설정과 순·역문제\n- 6개 방법, 기본 1시드·선택형 4시드\n- PDE RAD / 나머지 합성 학습점 LHS\n- 실제 데이터 CSV 연결\n- 독립 보간·시간 외삽\n- Adam/L-BFGS 분리 평가\n- 20% 간격 단일 재개 ZIP\n- 보고서·그림·API·도구형 질문·모델 등록\n\nOptuna는 포함하지 않습니다. 원통 후류는 외부 데이터가 필요합니다.\n"
    (root / "FEATURES.md").write_text(catalog, encoding="utf-8")
    return root / "REPORT.md"


def make_plots(root, trial_id=None, time_value=None, component=0):
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    root = Path(root)
    out = root / "plots"
    out.mkdir(exist_ok=True)
    rows = comparison(root)
    fig, ax = plt.subplots(figsize=(10, 4))
    for stage in ("adam", "lbfgs"):
        chosen = [r for r in rows if r["stage"] == stage and r["l2"] is not None]
        ax.scatter(range(len(chosen)), [r["l2"] * 100 for r in chosen], label=stage)
    ax.set_ylabel("Field L2 (%)")
    ax.set_xlabel("Saved trial index")
    if rows:
        ax.legend()
    fig.tight_layout()
    for suffix in ("png", "pdf"):
        fig.savefig(out / ("comparison." + suffix))
    plt.close(fig)
    if trial_id:
        from .config import Settings
        from .data import DataSource, make_plan
        from .runtime import Runtime
        from .equations import exact_field

        if trial_id not in {r["trial_id"] for r in reports(root)}:
            raise ValueError("완료 실험을 선택하세요.")
        trial = root / "trials" / trial_id
        cfg = json.loads((trial / "trial_settings.json").read_text(encoding="utf-8"))
        settings = Settings.from_dict(cfg["settings"])
        source = DataSource(
            settings, root / "inputs/data.json" if settings.data_mode == "csv" else None
        )
        rt = Runtime(settings, cfg["arm"], source)
        rt.load(trial / "selected.npz")
        plan = make_plan(settings, cfg["seed"], source.domain)
        x = plan.fixed_evaluation_points(32, 1)
        if plan.time_axis is not None:
            if time_value is None:
                time_value = source.domain[plan.time_axis][0]
            if (
                not source.domain[plan.time_axis][0]
                <= time_value
                <= source.domain[plan.time_axis][1]
            ):
                raise ValueError("시간 범위 오류")
            x[:, plan.time_axis] = time_value
        pred = rt.predict(x)
        exact = (
            exact_field(settings.equation, x)
            if settings.data_mode == "synthetic"
            else None
        )
        if type(component) is not int or not 0 <= component < pred.shape[1]:
            raise ValueError("출력 성분을 확인하세요.")
        physics = np.concatenate(
            [rt.pde_residual(x[i : i + 128]).numpy() for i in range(0, len(x), 128)]
        ) / np.asarray(rt.scales["residual"])
        columns = (
            [("Prediction", pred[:, component])]
            if exact is None
            else [
                ("Reference", exact[:, component]),
                ("Prediction", pred[:, component]),
                ("Absolute error", abs(pred[:, component] - exact[:, component])),
            ]
        )
        columns.append(
            ("Normalized residual RMS", np.sqrt(np.mean(physics**2, axis=1)))
        )
        fig, axes = plt.subplots(
            1, len(columns), figsize=(4 * len(columns), 4), squeeze=False
        )
        for ax, (title, z) in zip(axes[0], columns):
            if len(plan.spatial_axes) == 2:
                points = ax.scatter(
                    x[:, plan.spatial_axes[0]], x[:, plan.spatial_axes[1]], c=z, s=8
                )
                fig.colorbar(points, ax=ax)
            else:
                ax.plot(x[:, plan.spatial_axes[0]], z, ".")
            ax.set_title(title)
        fig.tight_layout()
        for suffix in ("png", "pdf"):
            fig.savefig(out / (trial_id + "_field." + suffix))
        plt.close(fig)
        history = json.loads((trial / "report.json").read_text(encoding="utf-8"))[
            "history"
        ]
        fig, ax = plt.subplots()
        ax.plot(
            [r["iteration"] for r in history],
            [r["validation"]["relative_l2"] for r in history],
        )
        ax.set_xlabel("Adam updates")
        ax.set_ylabel("Validation relative L2")
        fig.tight_layout()
        fig.savefig(out / (trial_id + "_history.png"))
        plt.close(fig)
        diagnostics = json.loads((trial / "report.json").read_text(encoding="utf-8"))[
            "diagnostics"
        ]
        if diagnostics:
            fig, axes = plt.subplots(2, 2, figsize=(10, 7))
            iterations = [r["iteration"] for r in diagnostics]
            for a, label in enumerate(("PDE", "data", "constraints")):
                axes[0, 0].plot(
                    iterations,
                    [r["raw_normalized_groups"][a] for r in diagnostics],
                    label=label,
                )
                axes[0, 1].plot(
                    iterations,
                    [r["network_gradient_norms"][a] for r in diagnostics],
                    label=label,
                )
                axes[1, 1].plot(
                    iterations,
                    [r["effective_weights"][a] for r in diagnostics],
                    label=label,
                )
            axes[1, 0].plot(
                iterations, [r["coefficient"] for r in diagnostics], label="coefficient"
            )
            for ax, title in zip(
                axes.flat,
                (
                    "Normalized losses",
                    "Gradient norms",
                    "Coefficient",
                    "Effective group weights",
                ),
            ):
                ax.set_title(title)
                ax.set_xlabel("Adam updates")
                ax.legend()
            fig.tight_layout()
            fig.savefig(out / (trial_id + "_diagnostics.png"))
            plt.close(fig)
    return sorted(out.glob("*"))


In [ ]:
%%writefile "{V4_MODULE_DIR}/runner.py"
"""Explicitly authorized, sequential V4 experiments with one rolling resume slot."""

import json
import hashlib
from pathlib import Path
import shutil
import tempfile
import time
import uuid
import numpy as np

from .config import checkpoint_due, digest, target_reached
from .data import DataSource, make_plan
from .metrics import evaluate, observed_metrics
from .runtime import Runtime
from .sampling import permitted_lhs, rad_select, rng_for
from .storage import arrays_sha, pack, restore, verify_zip, write_json


def training_source_digest():
    directory = Path(__file__).parent
    names = (
        "runner",
        "runtime",
        "config",
        "data",
        "equations",
        "generalization",
        "sampling",
        "metrics",
        "balance",
    )
    return digest(
        {
            name: hashlib.sha256(
                (directory / (name + ".py")).read_text(encoding="utf-8").encode()
            ).hexdigest()
            for name in names
        }
    )


def trial_identity(settings, arm, seed, source):
    return digest(
        {
            "settings": settings.to_dict(),
            "arm": arm,
            "seed": seed,
            "data": source.fingerprint,
        }
    )[:16]


def renew_pde(runtime, plan, cycle):
    s = runtime.settings
    pool = permitted_lhs(plan, s.rad_candidates, plan.seed, "pde", cycle)
    before = runtime.snapshot()
    scores = []
    for start in range(0, len(pool), 128):
        r = runtime.pde_residual(pool[start : start + 128]).numpy() / np.asarray(
            runtime.scales["residual"]
        )
        scores.append(np.sqrt(np.mean(r * r, axis=1)))
    runtime.assert_unchanged(before)
    selected, report = rad_select(
        pool,
        np.concatenate(scores),
        s.pde_count,
        rng_for(plan.seed, "rad_choice", cycle),
        s.rad_power,
        s.rad_offset,
    )
    report.update(
        cycle=cycle,
        candidate_sha=arrays_sha({"pool": pool}),
        selected_sha=arrays_sha({"pde": selected}),
    )
    plan.assert_training(selected, "pde")
    return selected, report


def save_state(trial, runtime, arrays, fixed, state):
    runtime.save(trial / "checkpoint.npz")
    np.savez_compressed(trial / "arrays.npz", **arrays)
    np.savez_compressed(trial / "fixed.npz", **fixed)
    state = dict(state)
    state.update(array_sha=arrays_sha(arrays), fixed_sha=arrays_sha(fixed))
    write_json(trial / "state.json", state)


def resume_slot(root, trial, output):
    with tempfile.TemporaryDirectory(prefix="v4_resume_", dir=root.parent) as temporary:
        scratch = Path(temporary)
        contract = json.loads((root / "contract.json").read_text(encoding="utf-8"))
        contract.update(kind="resume", trial_id=trial.name)
        write_json(scratch / "contract.json", contract)
        for name in (
            "checkpoint.npz",
            "arrays.npz",
            "fixed.npz",
            "state.json",
            "trial_settings.json",
        ):
            shutil.copyfile(trial / name, scratch / name)
        if (root / "inputs").exists():
            shutil.copytree(root / "inputs", scratch / "inputs")
        return pack(scratch, output)


def restore_slot(bundle, root, expected_id):
    owner = json.loads((root / "contract.json").read_text(encoding="utf-8"))
    with tempfile.TemporaryDirectory(
        prefix="v4_resume_read_", dir=root.parent
    ) as temporary:
        scratch = Path(temporary) / "state"
        restore(
            bundle,
            scratch,
            expected={
                "kind": "resume",
                "run_id": owner["run_id"],
                "trial_id": expected_id,
            },
        )
        trial = root / "trials" / expected_id
        trial.mkdir(parents=True, exist_ok=True)
        for name in (
            "checkpoint.npz",
            "arrays.npz",
            "fixed.npz",
            "state.json",
            "trial_settings.json",
        ):
            shutil.copyfile(scratch / name, trial / name)


def run_trial(
    root,
    settings,
    arm,
    seed,
    source,
    resume_zip=None,
    results_zip=None,
    log=print,
    update_limit=None,
):
    settings.validate()
    root = Path(root)
    plan = make_plan(settings, seed, source.domain)
    contract = json.loads((root / "contract.json").read_text(encoding="utf-8"))
    if contract.get("training_source_sha") != training_source_digest():
        raise ValueError("학습 코드가 저장 실행과 다릅니다. 새 독립 실험을 사용하세요.")
    identity = trial_identity(settings, arm, seed, source)
    trial = root / "trials" / identity
    if (trial / "report.json").is_file():
        report = json.loads((trial / "report.json").read_text(encoding="utf-8"))
        log("완료 실험 건너뜀: " + identity)
        return report
    if resume_zip and Path(resume_zip).is_file():
        contract = verify_zip(resume_zip)
        if contract.get("trial_id") == identity:
            restore_slot(resume_zip, root, identity)
    trial.mkdir(parents=True, exist_ok=True)
    expected = {
        "settings": settings.to_dict(),
        "arm": arm,
        "seed": seed,
        "source": source.fingerprint,
        "plan": plan.to_dict(),
    }
    expected_digest = digest(expected)
    runtime = Runtime(settings, arm, source)
    if (trial / "state.json").exists():
        state = json.loads((trial / "state.json").read_text(encoding="utf-8"))
        if state["contract_sha"] != expected_digest:
            raise ValueError("재개 설정·데이터·분할 계약 불일치")
        runtime.load(trial / "checkpoint.npz")
        with np.load(trial / "arrays.npz", allow_pickle=False) as z:
            arrays = {k: z[k].copy() for k in z.files}
        with np.load(trial / "fixed.npz", allow_pickle=False) as z:
            fixed = {k: z[k].copy() for k in z.files}
        if (
            arrays_sha(arrays) != state["array_sha"]
            or arrays_sha(fixed) != state["fixed_sha"]
        ):
            raise ValueError("재개 좌표 해시 불일치")
        if int(runtime.opt.iterations) != state["iteration"]:
            raise ValueError("optimizer iteration 불일치")
        for family in ("pde", "observations", "boundary", "initial", "gauge"):
            if family in arrays:
                plan.assert_training(arrays[family], family)
    else:
        arrays = source.training(plan, 0)
        if arm == "supervised_only":
            arrays["pde"] = permitted_lhs(plan, settings.pde_count, seed, "pde", 0)
            rad = []
        else:
            arrays["pde"], rad_record = renew_pde(runtime, plan, 0)
            rad = [rad_record]
        validation = source.validation(plan)
        evaluation = source.evaluation(plan)
        physics = permitted_lhs(
            plan, min(settings.evaluation_count, 1024), 93235, "pde", 0
        )
        # In extrapolation, field and PDE evaluation are both in the unseen time range.
        if plan.mode == "time_extrapolation":
            physics = plan.fixed_evaluation_points(16, 5)
        fixed = {
            "validation_x": validation["x"],
            "validation_y": validation["y"],
            "physics": physics,
        }
        constraint_arrays = source.training(plan, 93235)
        for name, value in constraint_arrays.items():
            if "observations" not in name:
                fixed["constraint_" + name] = value
        if evaluation is not None:
            fixed.update(evaluation_x=evaluation["x"], evaluation_y=evaluation["y"])
        state = {
            "contract_sha": expected_digest,
            "iteration": 0,
            "cycle": 0,
            "phase": "adam",
            "history": [],
            "diagnostics": [],
            "rad_history": rad,
            "training_seconds": 0.0,
            "observer_seconds": 0.0,
            "save_seconds": 0.0,
        }
        write_json(trial / "trial_settings.json", expected)
    tensors = runtime.prepare(arrays)
    updates = 0
    while state["phase"] == "adam" and state["iteration"] < settings.cap:
        if target_reached(state["history"], settings):
            break
        completed = state["iteration"]
        cycle = completed // settings.sampling_every
        if cycle != state["cycle"]:
            arrays = source.training(plan, cycle)
            if arm == "supervised_only":
                arrays["pde"] = permitted_lhs(
                    plan, settings.pde_count, seed, "pde", cycle
                )
            else:
                arrays["pde"], record = renew_pde(runtime, plan, cycle)
                state["rad_history"].append(record)
            tensors = runtime.prepare(arrays)
            state["cycle"] = cycle
        start = time.perf_counter()
        if (
            arm in {"loss_sa", "ff_loss_sa"}
            and completed
            and completed % settings.sa_every == 0
        ):
            runtime.sa_update(tensors)
        runtime.step(tensors, completed)
        state["training_seconds"] += time.perf_counter() - start
        state["iteration"] += 1
        updates += 1
        it = state["iteration"]
        if it % settings.check_every == 0 or it == settings.cap:
            start = time.perf_counter()
            validation = observed_metrics(
                fixed["validation_y"], runtime.predict(fixed["validation_x"])
            )
            state["history"].append(
                {
                    "iteration": it,
                    "validation": validation,
                    "coefficient": (
                        float(runtime.coefficient) if runtime.inverse else None
                    ),
                }
            )
            log(
                f"{settings.equation} {arm} {seed} {it} 검증 L1={validation['relative_l1']} L2={validation['relative_l2']}"
            )
            state["observer_seconds"] += time.perf_counter() - start
        if it % settings.diagnostic_every == 0 or it == settings.cap:
            start = time.perf_counter()
            diagnostic_arrays = {k: v[: min(len(v), 32)] for k, v in arrays.items()}
            state["diagnostics"].append(
                runtime.observe(runtime.prepare(diagnostic_arrays), it)
            )
            state["observer_seconds"] += time.perf_counter() - start
        if checkpoint_due(it, settings.cap) or (
            update_limit is not None and updates >= update_limit
        ):
            start = time.perf_counter()
            save_state(trial, runtime, arrays, fixed, state)
            if resume_zip:
                resume_slot(root, trial, resume_zip)
            state["save_seconds"] += time.perf_counter() - start
        if update_limit is not None and updates >= update_limit:
            return {
                "status": "interrupted_probe",
                "trial_id": identity,
                "iteration": it,
                "optimizer_updates_this_call": updates,
            }
    evaluation = (
        {"x": fixed["evaluation_x"], "y": fixed["evaluation_y"]}
        if "evaluation_x" in fixed
        else None
    )
    validation = {"x": fixed["validation_x"], "y": fixed["validation_y"]}
    constraints = {
        k.removeprefix("constraint_"): v
        for k, v in fixed.items()
        if k.startswith("constraint_")
    }
    if state["phase"] == "adam":
        state["phase"] = "adam_done"
        runtime.save(trial / "adam.npz")
        save_state(trial, runtime, arrays, fixed, state)
        if resume_zip:
            resume_slot(root, trial, resume_zip)
    runtime.load(
        trial / "adam.npz"
        if (trial / "adam.npz").exists()
        else trial / "checkpoint.npz"
    )
    final_evaluation_start = time.perf_counter()
    adam = evaluate(
        runtime, evaluation, fixed["physics"], state["iteration"], "adam", constraints
    )
    adam["validation"] = observed_metrics(
        validation["y"], runtime.predict(validation["x"])
    )
    adam["normalized_groups"] = (
        runtime.groups(tensors, arm != "supervised_only").numpy().tolist()
    )
    adam["optimizer_updates"] = state["iteration"]
    adam_evaluation_seconds = time.perf_counter() - final_evaluation_start
    state["phase"] = "adam_done"
    before = runtime.snapshot()
    start = time.perf_counter()
    lb = None
    solver_seconds = 0.0
    if settings.lbfgs_maxiter:
        solver = runtime.lbfgs(tensors, state["iteration"])
        solver_seconds = time.perf_counter() - start
        lb = evaluate(
            runtime,
            evaluation,
            fixed["physics"],
            state["iteration"],
            "lbfgs",
            constraints,
        )
        lb["validation"] = observed_metrics(
            validation["y"], runtime.predict(validation["x"])
        )
        lb["normalized_groups"] = (
            runtime.groups(tensors, arm != "supervised_only").numpy().tolist()
        )
        lb["solver"] = solver
        runtime.save(trial / "lbfgs.npz")
        a = adam["validation"]
        b = lb["validation"]
        selected = (
            solver["finite"]
            and solver["objective_after"] <= solver["objective_before"]
            and b["relative_l1"] is not None
            and b["relative_l2"] is not None
            and a["relative_l1"] is not None
            and a["relative_l2"] is not None
            and max(b["relative_l1"], b["relative_l2"])
            <= max(a["relative_l1"], a["relative_l2"])
        )
        if not selected:
            for v, value in zip(runtime.variables(), before[:-2]):
                v.assign(value)
            runtime.factors = before[-2].copy()
            runtime.sa_updates = int(before[-1])
    else:
        selected = False
    runtime.save(trial / "selected.npz")
    elapsed_lb = time.perf_counter() - start
    report = {
        "status": "complete",
        "trial_id": identity,
        "equation": settings.equation,
        "inverse": settings.inverse,
        "arm": arm,
        "seed": seed,
        "generalization": plan.summary(),
        "adam": adam,
        "lbfgs": lb,
        "selected_stage": "lbfgs" if selected else "adam",
        "stop_reason": (
            "target_reached"
            if target_reached(state["history"], settings)
            else "cap_reached"
        ),
        "target_reached": target_reached(state["history"], settings),
        "history": state["history"],
        "diagnostics": state["diagnostics"],
        "field_target": settings.target,
        "stop_policy": {
            "first": settings.stop_first,
            "every": settings.check_every,
            "consecutive": settings.consecutive,
            "both_l1_l2": True,
        },
        "execution_scope": (
            "disposable_smoke" if settings.cap < settings.stop_first else "research"
        ),
        "condition": {
            "noise_fraction": settings.noise_fraction,
            "initial_coefficient": settings.initial_coefficient,
        },
        "protocol_sha": digest(
            {
                "settings": {
                    k: v
                    for k, v in settings.to_dict().items()
                    if k not in {"arms", "seeds"}
                },
                "data": source.fingerprint,
            }
        ),
        "rad_history": state["rad_history"],
        "fixed_arrays_sha": arrays_sha(fixed),
        "training_arrays_sha": arrays_sha(arrays),
        "timing": {
            "adam_seconds": state["training_seconds"],
            "observer_seconds": state["observer_seconds"],
            "save_seconds": state["save_seconds"],
            "adam_final_evaluation_seconds": adam_evaluation_seconds,
            "lbfgs_solver_seconds": solver_seconds,
            "lbfgs_stage_seconds_including_evaluation": elapsed_lb,
        },
        "optimizer_updates_this_call": updates,
        "scales": runtime.scales,
        "runtime_versions": {
            "tensorflow": runtime.tf.__version__,
            "numpy": np.__version__,
        },
        "coefficient_identification": (
            "not_estimated"
            if arm == "supervised_only"
            else "single-positive-coefficient; sensitivity audit separate from trained identifiability"
        ),
    }
    write_json(trial / "report.json", report)
    if results_zip:
        from .reporting import generate_report

        generate_report(root)
        pack(root, results_zip)
    if resume_zip and Path(resume_zip).exists():
        receipt = verify_zip(resume_zip)
        owner = json.loads((root / "contract.json").read_text(encoding="utf-8"))
        if (
            results_zip
            and receipt.get("run_id") == owner["run_id"]
            and receipt.get("trial_id") == identity
        ):
            Path(
                resume_zip
            ).unlink()  # Only this verified, completed trial's single resume slot.
    return report


def create_run(root, settings, contract_path=None):
    settings.validate()
    root = Path(root)
    if root.exists():
        raise ValueError("새 실험 작업폴더가 필요합니다.")
    root.mkdir(parents=True)
    source = DataSource(settings, contract_path)
    if contract_path:
        inputs = root / "inputs"
        inputs.mkdir()
        original = Path(contract_path)
        copied = dict(source.contract)
        copied["files"] = {}
        for family, name in source.contract["files"].items():
            file = (original.parent / name).resolve()
            target = inputs / (family + ".csv")
            shutil.copyfile(file, target)
            copied["files"][family] = target.name
        write_json(inputs / "data.json", copied)
        source = DataSource(settings, inputs / "data.json")
    write_json(root / "settings.json", settings.to_dict())
    snapshot = root / "source_snapshot/v4"
    snapshot.mkdir(parents=True)
    for file in Path(__file__).parent.glob("*.py"):
        shutil.copyfile(file, snapshot / file.name)
    write_json(
        root / "contract.json",
        {
            "format": "v4-independent-1",
            "kind": "results",
            "run_id": uuid.uuid4().hex[:8],
            "settings_sha": digest(settings.to_dict()),
            "data_fingerprint": source.fingerprint,
            "training_source_sha": training_source_digest(),
        },
    )
    return source


def run_all(
    root,
    settings,
    source,
    results_zip,
    resume_zip,
    training_authorized=False,
    log=print,
):
    if training_authorized is not True:
        raise PermissionError("학습 실행 허용이 필요합니다.")
    if resume_zip and Path(resume_zip).is_file():
        receipt = verify_zip(resume_zip)
        owner = json.loads((Path(root) / "contract.json").read_text(encoding="utf-8"))
        if receipt["kind"] != "resume" or receipt["run_id"] != owner["run_id"]:
            raise ValueError("다른 실행의 재개 ZIP입니다.")
    completed = []
    for seed in settings.seeds:
        for arm in settings.arms:
            report = run_trial(
                root, settings, arm, seed, source, resume_zip, results_zip, log
            )
            completed.append(report)
            import tensorflow as tf

            tf.keras.backend.clear_session()
    if results_zip and not Path(results_zip).is_file():
        from .reporting import generate_report

        generate_report(root)
        pack(root, results_zip)
    return completed


In [ ]:
%%writefile "{V4_MODULE_DIR}/runtime.py"
"""Lazy TensorFlow PINN, separate optimizers, observer and frozen L-BFGS."""

import numpy as np
import random

from .config import group_weights, learning_rates
from .equations import EQUATIONS, residual


class State(list):
    """Model/optimizer state plus observer-only RNG guards, never a checkpoint."""

    pass


def load_tensorflow():
    try:
        import tensorflow as tf

        if not hasattr(tf, "GradientTape"):
            raise ImportError("incomplete TensorFlow package")
    except (ImportError, AttributeError) as exc:
        raise RuntimeError(
            "TensorFlow를 사용할 수 없습니다. 새 런타임에서 tensorflow 설치 후 다시 실행하세요. 버전 일치는 요구하지 않습니다."
        ) from exc
    for device in tf.config.list_physical_devices("GPU"):
        try:
            tf.config.experimental.set_memory_growth(device, True)
        except RuntimeError:
            pass  # A caller may already have initialized this device.
    return tf


class Runtime:
    def __init__(self, settings, arm, source):
        self.tf = tf = load_tensorflow()
        self.settings = settings
        self.arm = arm
        self.source = source
        self.scales = source.scales()
        spec = EQUATIONS[settings.equation]
        self.inverse = settings.inverse and arm != "supervised_only"
        initial = (
            settings.initial_coefficient
            if settings.initial_coefficient is not None
            else spec["initial"]
        )
        domain = np.array(source.domain, np.float32)
        field = np.array(self.scales["field"], np.float32)

        class PINN(tf.keras.Model):
            def __init__(self):
                super().__init__()
                self.log_coefficient = self.add_weight(
                    name="log_coefficient",
                    shape=(),
                    initializer=tf.keras.initializers.Constant(
                        np.log(
                            initial if settings.inverse else source.fixed_coefficient
                        )
                    ),
                    trainable=self_inverse,
                )
                self.banks = []
                rng = np.random.default_rng(4045)
                if arm.startswith("ff"):
                    for i, scale in enumerate(settings.ff_scales):
                        bank = rng.normal(
                            0, scale, (len(domain), settings.ff_features)
                        ).astype(np.float32)
                        self.banks.append(
                            self.add_weight(
                                name=f"ff_bank_{i}",
                                shape=bank.shape,
                                initializer=tf.keras.initializers.Constant(bank),
                                trainable=False,
                            )
                        )
                self.hidden = [
                    tf.keras.layers.Dense(
                        settings.width,
                        activation="tanh",
                        kernel_initializer=tf.keras.initializers.GlorotNormal(3234 + i),
                    )
                    for i in range(settings.depth)
                ]
                self.output_layer = tf.keras.layers.Dense(
                    len(spec["outputs"]),
                    kernel_initializer=tf.keras.initializers.GlorotNormal(
                        3234 + settings.depth
                    ),
                )

            def call(self, x, training=False):
                z = (
                    2
                    * (tf.cast(x, tf.float32) - domain[:, 0])
                    / (domain[:, 1] - domain[:, 0])
                    - 1
                )
                if self.banks:
                    phases = [2 * np.pi * tf.matmul(z, b) for b in self.banks]
                    z = tf.concat(
                        [f(p) for p in phases for f in (tf.sin, tf.cos)], axis=1
                    )
                for layer in self.hidden:
                    z = layer(z)
                return self.output_layer(z) * field

        self_inverse = self.inverse
        self.model = PINN()
        self.model(tf.zeros((1, len(domain))))
        self.network = [
            v
            for v in self.model.trainable_variables
            if v is not self.model.log_coefficient
        ]
        self.opt = tf.keras.optimizers.Adam(settings.base_lr)
        self.opt.build(self.network)
        self.coefficient_opt = tf.keras.optimizers.Adam(settings.base_lr * 0.001)
        self.coefficient_opt.build([self.model.log_coefficient])
        self.factors = np.ones(3, np.float32)
        self.sa_updates = 0
        self._step = tf.function(self._update, autograph=False, reduce_retracing=True)

    @property
    def coefficient(self):
        return (
            self.tf.exp(self.model.log_coefficient)
            if self.inverse
            else self.tf.constant(self.source.fixed_coefficient, self.tf.float32)
        )

    def derivatives(self, x, model=None):
        tf = self.tf
        x = tf.convert_to_tensor(x, tf.float32)
        model = model or self.model
        with tf.GradientTape(persistent=True) as outer:
            outer.watch(x)
            with tf.GradientTape(persistent=True) as inner:
                inner.watch(x)
                q = model(x)
                components = tf.unstack(q, axis=1)
            g = [
                inner.gradient(c, x, unconnected_gradients=tf.UnconnectedGradients.ZERO)
                for c in components
            ]
            parts = [tf.unstack(v, axis=1) for v in g]
        h = tf.stack(
            [
                tf.stack(
                    [
                        outer.gradient(
                            c, x, unconnected_gradients=tf.UnconnectedGradients.ZERO
                        )
                        for c in row
                    ],
                    axis=1,
                )
                for row in parts
            ],
            axis=1,
        )
        del inner, outer
        return q, tf.stack(g, axis=1), h

    def pde_residual(self, x, coefficient=None):
        q, g, h = self.derivatives(x)
        f = self.tf.constant(self.source.forcing(np.asarray(x)), self.tf.float32)
        return residual(
            self.settings.equation,
            self.tf.convert_to_tensor(x, self.tf.float32),
            q,
            g,
            h,
            self.coefficient if coefficient is None else coefficient,
            f,
            self.tf,
        )

    def tensor_data(self, arrays):
        return {
            k: self.tf.constant(np.nan_to_num(v), self.tf.float32)
            for k, v in arrays.items()
        }

    def prepare(self, arrays):
        data = dict(arrays)
        for key in list(arrays):
            if key.startswith("label_"):
                data["mask_" + key[6:]] = np.isfinite(arrays[key]).astype(np.float32)
        data["forcing"] = self.source.forcing(arrays["pde"])
        return self.tensor_data(data)

    def groups(self, a, include_physics=True):
        tf = self.tf
        fs = tf.constant(self.scales["field"], tf.float32)

        def observed(key, components=None):
            pred = self.model(a[key])
            label = a["label_" + key]
            mask = a["mask_" + key]
            if components is not None:
                pred = pred[:, :components]
                label = label[:, :components]
                mask = mask[:, :components]
                scale = fs[:components]
            else:
                scale = fs
            return tf.reduce_sum(tf.square((pred - label) / scale) * mask) / tf.maximum(
                tf.reduce_sum(mask), 1.0
            )

        if include_physics:
            q, g, h = self.derivatives(a["pde"])
            r = residual(
                self.settings.equation,
                a["pde"],
                q,
                g,
                h,
                self.coefficient,
                a["forcing"],
                tf,
            )
            physics = tf.reduce_mean(
                tf.square(r / tf.constant(self.scales["residual"], tf.float32))
            )
        else:
            physics = tf.constant(0.0, tf.float32)
        data = observed("observations")
        terms = []
        if "boundary" in a:
            terms.append(
                observed("boundary", 2 if len(self.scales["field"]) == 3 else None)
            )
        if "initial" in a:
            terms.append(observed("initial"))
        if "initial_velocity" in a:
            _, g, _ = self.derivatives(a.get("initial_velocity_x", a["initial"]))
            time_axis = EQUATIONS[self.settings.equation]["time_axis"]
            duration = np.ptp(np.asarray(self.source.domain)[time_axis])
            terms.append(
                tf.reduce_mean(
                    tf.square(
                        (g[:, 0, time_axis : time_axis + 1] - a["initial_velocity"])
                        * duration
                        / fs[0]
                    )
                )
            )
        if "gauge" in a:
            pred = self.model(a["gauge"])[:, 2:3]
            terms.append(
                tf.reduce_mean(tf.square((pred - a["label_gauge"][:, 2:3]) / fs[2]))
            )
        constraint = (
            tf.reduce_mean(tf.stack(terms)) if terms else tf.constant(0.0, tf.float32)
        )
        return tf.stack([physics, data, constraint])

    def _update(self, data, weights):
        tf = self.tf
        variables = self.network + (
            [self.model.log_coefficient] if self.inverse else []
        )
        with tf.GradientTape() as tape:
            objective = tf.reduce_sum(
                self.groups(data, self.arm != "supervised_only") * weights
            )
        grads = tape.gradient(objective, variables)
        grads = [tf.zeros_like(v) if g is None else g for g, v in zip(grads, variables)]
        for g in grads:
            tf.debugging.assert_all_finite(g, "Nonfinite training gradient")
        tf.debugging.assert_all_finite(objective, "Nonfinite objective")
        self.opt.apply_gradients(zip(grads[: len(self.network)], self.network))
        if self.inverse:
            self.coefficient_opt.apply_gradients(
                [(grads[-1], self.model.log_coefficient)]
            )
            self.model.log_coefficient.assign(
                tf.clip_by_value(self.model.log_coefficient, -20.0, 20.0)
            )
        return objective

    def step(self, tensors, completed):
        net, coef = learning_rates(completed, self.settings.base_lr)
        self.opt.learning_rate.assign(net)
        self.coefficient_opt.learning_rate.assign(coef)
        return float(
            self._step(
                tensors,
                self.tf.constant(group_weights(self.arm, completed, self.factors)),
            )
        )

    def variables(self):
        return (
            list(self.model.variables)
            + list(self.opt.variables)
            + list(self.coefficient_opt.variables)
        )

    def snapshot(self):
        state = State(
            [v.numpy().copy() for v in self.variables()]
            + [self.factors.copy(), np.array(self.sa_updates)]
        )
        state.python_rng = random.getstate()
        state.numpy_rng = np.random.get_state()
        state.tf_generator = self.tf.random.get_global_generator().state.numpy().copy()
        return state

    def assert_unchanged(self, before):
        after = self.snapshot()
        if len(before) != len(after) or not all(
            np.array_equal(a, b) for a, b in zip(before, after)
        ):
            raise RuntimeError("Observer changed model/optimizer state")
        if isinstance(before, State):
            if (
                before.python_rng != after.python_rng
                or not all(
                    np.array_equal(a, b)
                    for a, b in zip(before.numpy_rng, after.numpy_rng)
                )
                or not np.array_equal(before.tf_generator, after.tf_generator)
            ):
                raise RuntimeError(
                    "Observer changed Python/NumPy/global TensorFlow Generator state"
                )

    def term_gradients(self, tensors, include_coefficient=True):
        tf = self.tf
        with tf.GradientTape(persistent=True) as tape:
            terms = tf.unstack(self.groups(tensors))
        gradients = [tape.gradient(term, self.network) for term in terms]
        gradients = [
            [tf.zeros_like(v) if g is None else g for g, v in zip(row, self.network)]
            for row in gradients
        ]
        coefficient = [
            (
                tape.gradient(term, self.model.log_coefficient)
                if include_coefficient
                else None
            )
            for term in terms
        ]
        del tape
        vectors = [
            np.concatenate([g.numpy().ravel() for g in row]) for row in gradients
        ]
        return (
            terms,
            vectors,
            [
                0.0 if g is None else float(g) / float(self.coefficient)
                for g in coefficient
            ],
        )

    def sa_update(self, tensors):
        if self.arm not in {"loss_sa", "ff_loss_sa"}:
            raise ValueError("SA 비교군에서만 SA를 갱신합니다.")
        # This read-only derivative path is covered by development tests. Full
        # model/optimizer/RNG snapshots remain in scheduled observations.
        _, vectors, _ = self.term_gradients(tensors, include_coefficient=False)
        norms = np.array([np.linalg.norm(v) for v in vectors])
        goal = np.clip(
            norms.mean() / np.maximum(norms, 1e-12), *self.settings.sa_bounds
        )
        if not np.isfinite(goal).all():
            raise ValueError("SA gradient invalid")
        self.factors = self.settings.sa_ema * self.factors + (
            1 - self.settings.sa_ema
        ) * goal.astype(np.float32)
        self.sa_updates += 1

    def observe(self, tensors, iteration):
        before = self.snapshot()
        terms, vectors, coef = self.term_gradients(tensors)
        norms = np.array([np.linalg.norm(v) for v in vectors])
        cosine = np.diag((norms > 1e-30).astype(float))
        for a in range(3):
            for b in range(a):
                cosine[a, b] = cosine[b, a] = float(
                    vectors[a] @ vectors[b] / max(norms[a] * norms[b], 1e-30)
                )
        frozen = None
        if self.inverse:
            scale = np.asarray(self.scales["residual"])
            intercept = self.pde_residual(tensors["pde"], 0.0).numpy() / scale
            slope = self.pde_residual(tensors["pde"], 1.0).numpy() / scale - intercept
            denominator = float(np.mean(slope * slope))
            optimum = (
                -float(np.mean(intercept * slope)) / denominator
                if denominator > 1e-20
                else None
            )
            analytic = float(
                2 * np.mean((intercept + float(self.coefficient) * slope) * slope)
            )
            gradient_matches = bool(np.isclose(analytic, coef[0], rtol=1e-3, atol=1e-5))
            if not gradient_matches or not np.allclose(coef[1:], 0.0, atol=1e-8):
                raise RuntimeError("계수 gradient 의존성 검사 실패")
            frozen = {
                "unconstrained_optimum": optimum,
                "positive_domain_infimum": (
                    max(0.0, optimum) if optimum is not None else None
                ),
                "analytic_gradient": analytic,
                "gradient_matches_autodiff": gradient_matches,
                "data_constraint_coefficient_independent": True,
                "field_frozen_only": True,
            }
        self.assert_unchanged(before)
        return {
            "iteration": iteration,
            "raw_normalized_groups": [float(v) for v in terms],
            "network_gradient_norms": norms.tolist(),
            "gradient_cosines": cosine.tolist(),
            "coefficient_term_gradients": coef,
            "coefficient": float(self.coefficient),
            "frozen_field_coefficient": frozen,
            "gradient_cosine_defined": (
                (norms > 1e-30)[:, None] & (norms > 1e-30)[None, :]
            ).tolist(),
            "rng_guard_scope": "Python, NumPy legacy RNG, TensorFlow global Generator; deterministic tanh model",
            "effective_weights": group_weights(
                self.arm, iteration, self.factors
            ).tolist(),
            "sa_factors": self.factors.tolist(),
            "next_learning_rates": list(
                learning_rates(iteration, self.settings.base_lr)
            ),
            "state_guard": "PASS",
            "not_a_causal_proof": True,
        }

    def save(self, path):
        arrays = {f"v{i}": v.numpy() for i, v in enumerate(self.variables())}
        arrays.update(factors=self.factors, sa_updates=np.array(self.sa_updates))
        np.savez_compressed(path, **arrays)

    def load(self, path):
        with np.load(path, allow_pickle=False) as z:
            variables = self.variables()
            if set(z.files) != {
                *(f"v{i}" for i in range(len(variables))),
                "factors",
                "sa_updates",
            }:
                raise ValueError("Checkpoint variable inventory mismatch")
            for i, v in enumerate(variables):
                value = z[f"v{i}"]
                if tuple(v.shape) != value.shape or not np.isfinite(value).all():
                    raise ValueError("Checkpoint shape/finite check failed")
            for i, v in enumerate(variables):
                v.assign(z[f"v{i}"])
            self.factors = z["factors"].astype(np.float32)
            self.sa_updates = int(z["sa_updates"])
            if (
                self.factors.shape != (3,)
                or not np.isfinite(self.factors).all()
                or (self.factors <= 0).any()
                or self.sa_updates < 0
            ):
                raise ValueError("Checkpoint SA state invalid")

    def predict(self, x, batch=256):
        return np.concatenate(
            [self.model(x[i : i + batch]).numpy() for i in range(0, len(x), batch)]
        )

    def lbfgs(self, tensors, completed):
        from scipy.optimize import minimize

        tf = self.tf
        variables = self.network + (
            [self.model.log_coefficient] if self.inverse else []
        )
        shapes = [tuple(v.shape) for v in variables]
        sizes = [int(np.prod(s)) for s in shapes]

        def vector():
            return np.concatenate([v.numpy().ravel() for v in variables]).astype(float)

        def assign(position):
            if not np.isfinite(position).all():
                raise ValueError("Nonfinite L-BFGS position")
            offset = 0
            for v, shape, size in zip(variables, shapes, sizes):
                v.assign(position[offset : offset + size].reshape(shape))
                offset += size

        weights = tf.constant(group_weights(self.arm, completed, self.factors))
        x0 = vector()
        calls = 0

        def objective(position):
            nonlocal calls
            calls += 1
            assign(position)
            with tf.GradientTape() as tape:
                loss = tf.reduce_sum(
                    self.groups(tensors, self.arm != "supervised_only") * weights
                )
            grads = tape.gradient(loss, variables)
            grads = [
                tf.zeros_like(v) if g is None else g for g, v in zip(grads, variables)
            ]
            grad = np.concatenate([v.numpy().ravel() for v in grads]).astype(float)
            value = float(loss)
            if not np.isfinite(value) or not np.isfinite(grad).all():
                raise ValueError("Nonfinite L-BFGS objective")
            return value, grad

        baseline, _ = objective(x0)
        try:
            result = minimize(
                objective,
                x0,
                jac=True,
                method="L-BFGS-B",
                options={
                    "maxiter": self.settings.lbfgs_maxiter,
                    "maxls": 20,
                    "maxfun": max(20, self.settings.lbfgs_maxiter * 30),
                    "ftol": 1e-12,
                    "gtol": 1e-8,
                },
            )
            assign(result.x)
            candidate, _ = objective(result.x)
            return {
                "iterations": int(result.nit),
                "function_evaluations": calls,
                "converged": bool(result.success),
                "message": str(result.message),
                "objective_before": baseline,
                "objective_after": candidate,
                "finite": True,
                "frozen_sampling_and_sa": True,
            }
        except (ValueError, FloatingPointError) as exc:
            assign(x0)
            return {
                "iterations": None,
                "function_evaluations": calls,
                "converged": False,
                "message": str(exc),
                "objective_before": baseline,
                "objective_after": None,
                "finite": False,
                "frozen_sampling_and_sa": True,
            }


In [ ]:
%%writefile "{V4_MODULE_DIR}/sampling.py"
"""Keyed LHS and RAD. Only PDE points use model-dependent adaptation."""

import hashlib
import numpy as np


def rng_for(seed, family, cycle=0):
    key = f"{seed}:{family}:{cycle}".encode()
    return np.random.default_rng(
        int.from_bytes(hashlib.sha256(key).digest()[:8], "little")
    )


def lhs(bounds, count, rng):
    b = np.asarray(bounds, float)
    if count < 1:
        return np.empty((0, len(b)))
    unit = np.column_stack(
        [(rng.permutation(count) + rng.random(count)) / count for _ in b]
    )
    return b[:, 0] + unit * (b[:, 1] - b[:, 0])


def permitted_lhs(plan, count, seed, family, cycle):
    rng = rng_for(seed, family, cycle)
    result = []
    remaining = count
    for _ in range(32):
        points = lhs(plan.training_domain, max(remaining, 16), rng)
        points = points[plan.training_mask(points, family)]
        result.append(points[:remaining])
        remaining -= min(remaining, len(points))
        if not remaining:
            values = np.concatenate(result)
            plan.assert_training(values, family)
            return values
    raise ValueError("학습 영역에 필요한 좌표를 확보하지 못했습니다.")


def rad_select(pool, scores, count, rng, power=1.0, offset=1.0):
    scores = np.asarray(scores, float)
    if (
        scores.ndim != 1
        or len(scores) != len(pool)
        or len(pool) < count
        or not np.isfinite(scores).all()
        or np.any(scores < 0)
    ):
        raise ValueError("RAD 점수·후보 수 오류")
    maximum = scores.max(initial=0)
    scaled = (scores / maximum) ** power if maximum else np.zeros_like(scores)
    mean = scaled.mean()
    weights = (scaled / mean if mean else np.zeros_like(scores)) + offset
    probabilities = weights / weights.sum()
    indices = rng.choice(len(pool), size=count, replace=False, p=probabilities)
    return np.asarray(pool)[indices].copy(), {
        "power": power,
        "offset": offset,
        "candidate_count": len(pool),
        "score_max": float(maximum),
        "uniform_probability_mass": float(offset / (1 + offset)) if mean else 1.0,
    }


In [ ]:
%%writefile "{V4_MODULE_DIR}/services.py"
"""Read-only API, hybrid RAG, tool-routed questions and manual inference registry."""

import json
from pathlib import Path
import re
import numpy as np

from .config import Settings, ARM_LABELS
from .data import DataSource
from .equations import EQUATIONS
from .reporting import comparison, generate_report, reports
from .storage import write_json, sha_file


class Service:
    def __init__(self, root):
        self.root = Path(root)
        self.cache = {}
        self.registry_path = self.root / "registry.json"

    def models(self):
        return [
            {
                "trial_id": r["trial_id"],
                "equation": r["equation"],
                "inverse": r["inverse"],
                "arm": r["arm"],
                "seed": r["seed"],
                "selected_stage": r["selected_stage"],
                "outputs": EQUATIONS[r["equation"]]["outputs"],
                "generalization": r["generalization"],
                "checkpoint_exists": (
                    self.root / "trials" / r["trial_id"] / "selected.npz"
                ).exists(),
            }
            for r in reports(self.root)
        ]

    def model_info(self, trial_id):
        found = next((r for r in self.models() if r["trial_id"] == trial_id), None)
        if found is None:
            raise ValueError("완료 모델이 없습니다.")
        cfg = json.loads(
            (self.root / "trials" / trial_id / "trial_settings.json").read_text(
                encoding="utf-8"
            )
        )
        spec = EQUATIONS[found["equation"]]
        source = DataSource(
            Settings.from_dict(cfg["settings"]),
            (
                self.root / "inputs/data.json"
                if cfg["settings"]["data_mode"] == "csv"
                else None
            ),
        )
        return {
            **found,
            "input_variables": [
                {"symbol": a, "range": d} for a, d in zip(spec["axes"], source.domain)
            ],
            "output_variables": spec["outputs"],
            "coefficient": spec["coefficient"],
            "settings": cfg["settings"],
        }

    def predict(self, trial_id, coordinates):
        from .runtime import Runtime

        info = self.model_info(trial_id)
        points = np.asarray(coordinates, float)
        if (
            points.ndim != 2
            or not 1 <= len(points) <= 1024
            or points.shape[1] != len(info["input_variables"])
            or not np.isfinite(points).all()
        ):
            raise ValueError("예측 좌표 차원·크기를 확인하세요.")
        domain = np.array([v["range"] for v in info["input_variables"]])
        if np.any(points < domain[:, 0]) or np.any(points > domain[:, 1]):
            raise ValueError("저장 영역 밖의 좌표입니다.")
        if trial_id not in self.cache:
            settings = Settings.from_dict(info["settings"])
            source = DataSource(
                settings,
                self.root / "inputs/data.json" if settings.data_mode == "csv" else None,
            )
            rt = Runtime(settings, info["arm"], source)
            rt.load(self.root / "trials" / trial_id / "selected.npz")
            self.cache[trial_id] = rt
        return {
            "trial_id": trial_id,
            "prediction": self.cache[trial_id].predict(points).tolist(),
            "outputs": info["output_variables"],
        }

    def registry(self):
        return (
            json.loads(self.registry_path.read_text(encoding="utf-8"))
            if self.registry_path.exists()
            else {"active": {}, "history": []}
        )

    def register(self, equation, trial_id, approved=False):
        if approved is not True:
            raise PermissionError("모델 변경 승인이 필요합니다.")
        info = self.model_info(trial_id)
        if info["equation"] != equation:
            raise ValueError("방정식 불일치")
        if not info["checkpoint_exists"]:
            raise ValueError("완료 모델 파일이 없습니다.")
        center = [[sum(v["range"]) / 2 for v in info["input_variables"]]]
        prediction = self.predict(trial_id, center)
        if not np.isfinite(prediction["prediction"]).all():
            raise ValueError("추론 모델 사전 검사 실패")
        validation = {
            "checkpoint_sha256": sha_file(
                self.root / "trials" / trial_id / "selected.npz"
            ),
            "settings_sha256": sha_file(
                self.root / "trials" / trial_id / "trial_settings.json"
            ),
            "model_load_and_finite_prediction": True,
        }
        state = self.registry()
        previous = state["active"].get(equation)
        state["history"].append(
            {
                "equation": equation,
                "previous": previous,
                "selected": trial_id,
                "validation": validation,
            }
        )
        state["active"][equation] = trial_id
        write_json(self.registry_path, state)
        return state

    def rollback(self, equation, approved=False):
        if approved is not True:
            raise PermissionError("모델 변경 승인이 필요합니다.")
        state = self.registry()
        item = next(
            (v for v in reversed(state["history"]) if v["equation"] == equation), None
        )
        if item is None or item["previous"] is None:
            raise ValueError("복원할 직전 모델이 없습니다.")
        return self.register(equation, item["previous"], True)

    def search(self, query, dense=False, backend="numpy", count=5):
        if (
            backend not in {"numpy", "faiss"}
            or not isinstance(query, str)
            or not 1 <= len(query) <= 2000
            or type(count) is not int
            or not 1 <= count <= 20
        ):
            raise ValueError("검색 입력·개수·엔진을 확인하세요.")
        documents = []
        for p in [self.root / "REPORT.md", self.root / "FEATURES.md"]:
            if p.exists():
                text = p.read_text(encoding="utf-8")
                documents.extend(
                    {"source": p.name, "text": text[i : i + 800]}
                    for i in range(0, len(text), 600)
                )
        if not documents:
            return []
        tokenize = lambda s: re.findall(r"[\w]+", s.lower())
        words = [tokenize(d["text"]) for d in documents]
        terms = tokenize(query)
        n = len(words)
        length = np.array([len(v) for v in words])
        average = max(float(length.mean()), 1.0)
        scores = np.zeros(n)
        for term in terms:
            frequency = np.array([v.count(term) for v in words])
            df = sum(term in v for v in words)
            idf = np.log(1 + (n - df + 0.5) / (df + 0.5))
            scores += (
                idf
                * frequency
                * 2.5
                / (frequency + 1.5 * (0.25 + 0.75 * length / average))
            )
        order = np.argsort(-scores)
        ranks = 0.4 / (60 + np.argsort(order) + 1)
        if dense:
            from sentence_transformers import SentenceTransformer

            if not hasattr(self, "embedding"):
                self.embedding = SentenceTransformer("intfloat/multilingual-e5-small")
            embeddings = np.asarray(
                self.embedding.encode(
                    ["passage: " + d["text"] for d in documents],
                    normalize_embeddings=True,
                ),
                np.float32,
            )
            q = np.asarray(
                self.embedding.encode(["query: " + query], normalize_embeddings=True),
                np.float32,
            )
            if backend == "faiss":
                import faiss

                index = faiss.IndexFlatIP(embeddings.shape[1])
                index.add(embeddings)
                _, indices = index.search(q, n)
                dense_order = indices[0]
            elif backend == "numpy":
                dense_order = np.argsort(-(embeddings @ q[0]))
            else:
                raise ValueError("검색 엔진 오류")
            ranks += 0.6 / (60 + np.argsort(dense_order) + 1)
        return [
            {
                **documents[i],
                "score": float(ranks[i]),
                "backend": "BM25" + (" + E5/" + backend + " + RRF" if dense else ""),
            }
            for i in np.argsort(-ranks)[:count]
        ]

    def tool(self, name, arguments=None, approved=False):
        arguments = arguments or {}
        if name == "comparison":
            return comparison(self.root)
        if name == "models":
            return self.models()
        if name == "model_info":
            return self.model_info(arguments.get("trial_id", ""))
        if name == "predict":
            return self.predict(
                arguments.get("trial_id", ""), arguments.get("coordinates", [])
            )
        if name == "search":
            return self.search(**arguments)
        if name in {"generate_report", "generate_feature_catalog"}:
            if approved is not True:
                raise PermissionError("문서 생성 승인이 필요합니다.")
            generate_report(self.root)
            file = self.root / (
                "REPORT.md" if name == "generate_report" else "FEATURES.md"
            )
            return {
                "text": file.read_text(encoding="utf-8"),
                "rag_reindex": "next search reads current generated documents",
            }
        raise ValueError("허용되지 않은 도구입니다.")

    def ask(self, question, use_llm=False, dense=False, backend="numpy"):
        if not isinstance(question, str) or not 1 <= len(question) <= 2000:
            raise ValueError("질문 길이는 1~2000자입니다.")
        if re.search(
            r"삭제|덮어|비밀번호|토큰|delete|overwrite|학습.*실행", question, re.I
        ):
            return {
                "answer": "질문으로 파일 변경이나 학습을 실행하지 않습니다. 해당 실행 패널에서 선택하세요.",
                "tool": "refuse",
                "llm_called": False,
            }
        equations = [
            k
            for k, v in EQUATIONS.items()
            if k.lower() in question.lower() or v["label"] in question
        ]
        aliases = {
            "포아송": "poisson2d",
            "파동": "wave2d",
            "열전도": "heat2d",
            "버거스": "burgers",
            "Darcy": "darcy2d",
            "반응": "reaction_diffusion",
            "Helmholtz": "helmholtz",
        }
        equations += [v for k, v in aliases.items() if k.lower() in question.lower()]
        if len(set(equations)) > 1:
            return {
                "answer": "방정식을 하나 지정해 주세요. 방정식마다 조건을 구분해서 비교합니다.",
                "tool": "clarify",
                "llm_called": False,
            }
        equation = next(iter(dict.fromkeys(equations)), None)
        called = False
        if re.search(r"예측|predict", question, re.I):
            match = re.search(r"\[\s*\[.*?\]\s*\]", question)
            active = self.registry()["active"].get(equation)
            if match and active:
                prediction = self.predict(active, json.loads(match.group()))
                return {
                    "answer": "저장 모델의 예측값은 "
                    + json.dumps(prediction["prediction"], ensure_ascii=False)
                    + "입니다.",
                    "tool": "predict",
                    "evidence": prediction,
                    "llm_called": False,
                }
            return {
                "answer": "좌표 JSON과 방정식을 지정하고 기본 추론 모델을 선택해 주세요.",
                "tool": "clarify",
                "llm_called": False,
            }
        if use_llm:
            from transformers import AutoTokenizer, AutoModelForCausalLM

            if not hasattr(self, "llm"):
                import torch

                self.tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3.5-0.8B")
                self.llm = AutoModelForCausalLM.from_pretrained(
                    "Qwen/Qwen3.5-0.8B", dtype="auto"
                )
                self.llm.to("cuda" if torch.cuda.is_available() else "cpu")
                self.llm.eval()
            prompt = (
                'Return JSON only: {"equation": supported_name_or_null, "tool": "comparison" or "search"}. No commands. Supported: '
                + ",".join(EQUATIONS)
                + "\nQuestion: "
                + question
            )
            messages = [{"role": "user", "content": prompt}]
            encoded = self.tokenizer.apply_chat_template(
                messages,
                return_tensors="pt",
                return_dict=True,
                add_generation_prompt=True,
                enable_thinking=False,
            ).to(self.llm.device)
            import torch

            with torch.inference_mode():
                generated = self.llm.generate(
                    **encoded, max_new_tokens=96, do_sample=False
                )
            text = self.tokenizer.decode(
                generated[0][encoded["input_ids"].shape[-1] :], skip_special_tokens=True
            )
            called = True
            try:
                proposed = json.loads(text[text.index("{") : text.rindex("}") + 1])
                if equation is None and proposed.get("equation") in EQUATIONS:
                    equation = proposed["equation"]
            except (ValueError, TypeError):
                pass
        rows = [
            r
            for r in comparison(self.root)
            if equation is None or r["equation"] == equation
        ]
        if equation and not rows:
            return {
                "answer": EQUATIONS[equation]["label"]
                + "의 완료된 V4 비교 실험이 아직 없습니다.",
                "tool": "comparison",
                "evidence": [],
                "llm_called": called,
            }
        if equation and rows:
            requested_stage = (
                "lbfgs" if re.search(r"L.?BFGS|후처리", question, re.I) else "adam"
            )
            if "역문제" in question:
                rows = [r for r in rows if r["inverse"]]
            elif "순방향" in question:
                rows = [r for r in rows if not r["inverse"]]
            rows = [
                r for r in rows if r["stage"] == requested_stage and r["l2"] is not None
            ]
            if rows:
                partitions = {
                    (
                        r["inverse"],
                        r["generalization"],
                        r["execution_scope"],
                        r["noise_fraction"],
                        r["initial_coefficient"],
                        r["protocol_sha"],
                    )
                    for r in rows
                }
                if len(partitions) > 1:
                    return {
                        "answer": "순·역문제와 기본·보간·외삽 실험을 구분해서 비교합니다. 원하는 범위를 지정해 주세요.",
                        "tool": "clarify",
                        "llm_called": called,
                    }
                best = min(rows, key=lambda r: r["l2"])
                answer = f"{EQUATIONS[equation]['label']}의 저장 {requested_stage} 결과에서 물리장 L2 오차가 가장 낮은 개별 실험은 {ARM_LABELS[best['arm']]} / 시드 {best['seed']}이며 {best['l2']*100:.5f}%입니다. 저장된 결과 비교입니다."
                return {
                    "answer": answer,
                    "tool": "comparison",
                    "evidence": rows,
                    "llm_called": called,
                }
            return {
                "answer": "요청한 방향·단계의 완료 결과가 없습니다.",
                "tool": "comparison",
                "evidence": [],
                "llm_called": called,
            }
        context = self.search(question, dense, backend)
        return {
            "answer": (
                "확인할 방정식·순/역문제·Adam/L-BFGS 단계를 지정해 주세요."
                if not rows
                else f"완료된 V4 실험 {len(reports(self.root))}개가 있습니다. 방정식을 지정하면 비교할 수 있습니다."
            ),
            "tool": "search",
            "context": context,
            "llm_called": called,
        }


def create_app(service, allow_registry_write=False):
    from .api_routes import build_app

    return build_app(service, allow_registry_write)


In [ ]:
%%writefile "{V4_MODULE_DIR}/settings_panel.py"
"""Described per-variable controls; serialization stays internal."""

from .config import Settings


def settings_editor():
    import ipywidgets as w

    specs = {
        "학습과 중단": [
            (
                "cap",
                "Adam 최대 반복",
                "각 방법·시드의 총 업데이트 한도입니다. 목표 미달이어도 이 한도에서 종료합니다.",
            ),
            (
                "target",
                "목표 오차 (%)",
                "고정 검증점의 L1과 L2가 모두 이 백분율 이하여야 합니다. 5를 입력하면 5%입니다.",
            ),
            (
                "stop_first",
                "중단 허용 시작",
                "이 반복부터 목표 달성에 의한 조기중단을 허용합니다.",
            ),
            (
                "check_every",
                "목표 확인 간격",
                "이 반복 간격마다 고정 검증점에서 목표를 확인합니다.",
            ),
            (
                "consecutive",
                "연속 만족 횟수",
                "목표를 연속으로 이 횟수만큼 만족해야 Adam을 종료합니다.",
            ),
            (
                "diagnostic_every",
                "진단 기록 간격",
                "손실별 기울기 등 진단을 기록하는 간격입니다. ZIP 저장 간격과 다릅니다.",
            ),
            (
                "base_lr",
                "기준 학습률",
                "시작 신경망 학습률입니다. 신경망/계수 비율은 0·5천·1만 회에서 1/0.001, 0.75/0.75, 0.5/1로 변합니다.",
            ),
            (
                "initial_coefficient",
                "역문제 초기 계수",
                "자동은 방정식별 기본값입니다. 직접 입력은 양수만 허용하며 관측값 회귀에는 적용되지 않습니다.",
            ),
            (
                "lbfgs_maxiter",
                "L-BFGS 최대 반복",
                "Adam 뒤 후처리 반복 한도입니다. 0이면 생략하며 함수 평가 횟수와는 다릅니다.",
            ),
        ],
        "신경망과 푸리에 특징": [
            ("width", "은닉층 뉴런 수", "각 은닉층의 너비입니다. 기본값은 64입니다."),
            ("depth", "은닉층 수", "tanh 은닉층의 개수입니다. 기본값은 5입니다."),
            (
                "ff_features",
                "FF 뱅크별 특징 수",
                "각 주파수 뱅크의 수입니다. 뱅크마다 sin/cos가 생성됩니다. FF 방법에서만 사용합니다.",
            ),
            (
                "ff_scales",
                "FF 주파수 척도",
                "쉼표로 구분한 양수입니다. 각 값이 한 뱅크의 무작위 주파수 표준편차입니다. 예: 0.125, 0.25, 0.5",
            ),
        ],
        "학습점과 고정 평가점": [
            (
                "sampling_every",
                "학습점 갱신 간격",
                "PDE는 RAD, 나머지 합성 학습점은 LHS로 갱신합니다. CSV 관측·조건점은 유지합니다.",
            ),
            (
                "pde_count",
                "PDE 학습점 수",
                "매 주기에 사용할 PDE 좌표 수입니다. 관측값 회귀에는 사용하지 않습니다.",
            ),
            (
                "observation_count",
                "합성 관측점 수",
                "합성 참조값으로 생성할 학습 관측점 수입니다. CSV 행 수를 바꾸지 않습니다.",
            ),
            (
                "boundary_count",
                "경계면당 점 수",
                "합성 문제의 각 공간 경계면에 생성하는 점 수입니다.",
            ),
            (
                "initial_count",
                "초기조건 점 수",
                "합성 시간 의존 문제의 초기 시점 좌표 수입니다. 정상 문제에는 적용하지 않습니다.",
            ),
            (
                "validation_count",
                "고정 검증점 수",
                "합성 문제의 중단 판정용 고정 좌표 수입니다. CSV는 연결한 검증 행을 사용합니다.",
            ),
            (
                "evaluation_count",
                "고정 평가점 수",
                "합성 문제의 결과 보고용 고정 좌표 수입니다. 학습·중단 판정에는 쓰지 않습니다.",
            ),
            (
                "noise_fraction",
                "합성 관측 노이즈 (%)",
                "특성 물리장 크기에 대한 Gaussian 노이즈 표준편차입니다. 0이면 노이즈를 추가하지 않습니다.",
            ),
        ],
        "RAD와 손실항 SA": [
            (
                "rad_candidates",
                "RAD 후보점 수",
                "PDE 학습점을 선택할 후보 좌표 수이며 PDE 학습점 수 이상이어야 합니다.",
            ),
            (
                "rad_power",
                "RAD 잔차 지수",
                "큰 PDE 잔차에 대한 선택 편향을 조절합니다. 기본값 1입니다.",
            ),
            (
                "rad_offset",
                "RAD 균일 혼합값",
                "균일 선택 성분입니다. 값이 커질수록 잔차 집중이 약해집니다. 기본값 1입니다.",
            ),
            (
                "sa_every",
                "SA 갱신 간격",
                "손실항 SA 방법에서 그룹 가중치를 조절하는 반복 간격입니다.",
            ),
            (
                "sa_ema",
                "SA 평활 계수",
                "0 이상 1 미만입니다. 클수록 이전 가중치를 더 유지합니다.",
            ),
            (
                "sa_bounds",
                "SA 배율 하한·상한",
                "쉼표로 구분한 두 양수입니다. 예: 0.25, 4.0. 정규화 전 그룹 배율의 제한입니다.",
            ),
        ],
        "보간 제외 구역": [
            (
                "block_override",
                "제외 구역 번호",
                "자동은 시드로 16구역 중 하나를 선택합니다. 직접 선택은 0~15이며 보간 실험에만 적용합니다.",
            ),
        ],
    }

    class Editor(w.VBox):
        def __init__(self):
            self.controls = {}
            self.auto = {}
            self._disabled = False
            self.context = {}
            self.stored = Settings().to_dict()
            pages = []
            for section, rows in specs.items():
                boxes = []
                for name, label, help_text in rows:
                    value = self.stored[name]
                    options = dict(
                        description=label,
                        style={"description_width": "170px"},
                        layout=w.Layout(width="100%"),
                    )
                    if name in {"initial_coefficient", "block_override"}:
                        self.auto[name] = w.Checkbox(
                            description="자동 선택", value=True
                        )
                        control = (
                            w.IntText(value=0, **options)
                            if name == "block_override"
                            else w.FloatText(value=0.5, **options)
                        )
                        self.auto[name].observe(self._auto_changed, names="value")
                    elif name in {"ff_scales", "sa_bounds"}:
                        control = w.Text(
                            value=", ".join(str(v) for v in value), **options
                        )
                    elif isinstance(value, int):
                        control = w.IntText(value=value, **options)
                    else:
                        control = w.FloatText(
                            value=(
                                value * 100
                                if name in {"target", "noise_fraction"}
                                else value
                            ),
                            **options,
                        )
                    self.controls[name] = control
                    boxes.append(
                        w.VBox(
                            [
                                w.HTML(f"<small>{name} — {help_text}</small>"),
                                *([self.auto[name]] if name in self.auto else []),
                                control,
                            ]
                        )
                    )
                pages.append(w.VBox(boxes))
            accordion = w.Accordion(children=pages)
            for index, title in enumerate(specs):
                accordion.set_title(index, title)
            accordion.selected_index = None
            super().__init__(
                [
                    w.HTML(
                        "항목을 펼쳐 값을 변경하세요. 재개 시 저장 설정을 그대로 불러옵니다. 변경한 설정은 새 독립 실험에 사용하세요."
                    ),
                    accordion,
                ]
            )
            self._apply_enabled()

        def _auto_changed(self, _=None):
            self._apply_enabled()

        def set_context(self, arms, inverse, generalization, data_mode, timed):
            self.context = dict(
                arms=arms,
                inverse=inverse,
                generalization=generalization,
                data_mode=data_mode,
                timed=timed,
            )
            self._apply_enabled()

        def _apply_enabled(self):
            c = self.context
            arms = c.get("arms", ())
            inactive = set()
            if c:
                if not any(k.startswith("ff") for k in arms):
                    inactive.update(("ff_features", "ff_scales"))
                if not any(k in {"loss_sa", "ff_loss_sa"} for k in arms):
                    inactive.update(("sa_every", "sa_ema", "sa_bounds"))
                if not any(k != "supervised_only" for k in arms):
                    inactive.update(
                        ("pde_count", "rad_candidates", "rad_power", "rad_offset")
                    )
                if not c["inverse"] or not any(k != "supervised_only" for k in arms):
                    inactive.add("initial_coefficient")
                if c["generalization"] != "interpolation":
                    inactive.add("block_override")
                if c["data_mode"] == "csv":
                    inactive.update(
                        (
                            "observation_count",
                            "boundary_count",
                            "initial_count",
                            "validation_count",
                            "evaluation_count",
                            "noise_fraction",
                        )
                    )
                if not c["timed"]:
                    inactive.add("initial_count")
            for name, control in self.controls.items():
                control.disabled = (
                    self._disabled
                    or name in inactive
                    or (name in self.auto and self.auto[name].value)
                )
            for name, control in self.auto.items():
                control.disabled = self._disabled or name in inactive

        @property
        def value(self):
            values = dict(self.stored)
            for name, control in self.controls.items():
                value = control.value
                if name in self.auto and self.auto[name].value:
                    value = None
                elif name in {"ff_scales", "sa_bounds"}:
                    try:
                        value = tuple(float(v.strip()) for v in value.split(","))
                    except ValueError as exc:
                        raise ValueError(
                            name + ": 쉼표로 구분한 숫자를 입력하세요."
                        ) from exc
                elif name in {"target", "noise_fraction"}:
                    value /= 100
                values[name] = value
            return values

        @value.setter
        def value(self, values):
            self.stored = dict(values)
            for name, control in self.controls.items():
                value = values[name]
                if name in self.auto:
                    self.auto[name].value = value is None
                    if value is None:
                        continue
                if name in {"ff_scales", "sa_bounds"}:
                    value = ", ".join(str(v) for v in value)
                elif name in {"target", "noise_fraction"}:
                    value *= 100
                control.value = value
            self._apply_enabled()

        @property
        def disabled(self):
            return self._disabled

        @disabled.setter
        def disabled(self, value):
            self._disabled = value
            self._apply_enabled()

    return Editor()


In [ ]:
%%writefile "{V4_MODULE_DIR}/storage.py"
"""Manifest-checked archives. Never executes source from downloaded results."""

import hashlib
import json
import os
from pathlib import Path, PurePosixPath
import stat
import tempfile
import zipfile


def sha_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    handle, temp = tempfile.mkstemp(
        prefix=path.name + ".", suffix=".tmp", dir=path.parent
    )
    try:
        with os.fdopen(handle, "w", encoding="utf-8") as stream:
            json.dump(value, stream, ensure_ascii=False, indent=2, allow_nan=False)
            stream.write("\n")
            stream.flush()
            os.fsync(stream.fileno())
        os.replace(temp, path)
    finally:
        if Path(temp).exists():
            Path(temp).unlink()


def verify_zip(path):
    with zipfile.ZipFile(path) as z:
        names = z.namelist()
        if len(names) != len(set(names)):
            raise ValueError("ZIP 항목 중복")
        for item in z.infolist():
            p = PurePosixPath(item.filename)
            if (
                p.is_absolute()
                or ".." in p.parts
                or "\\" in item.filename
                or ":" in item.filename
                or "\x00" in item.filename
                or stat.S_ISLNK(item.external_attr >> 16)
            ):
                raise ValueError("ZIP 경로가 안전하지 않습니다.")
        if sum(i.file_size for i in z.infolist()) > 4 * 1024**3:
            raise ValueError("ZIP 크기 한도 초과")
        manifest = json.loads(z.read("v4_manifest.json"))
        files = {i.filename for i in z.infolist() if not i.is_dir()}
        if files != set(manifest["files"]) | {"v4_manifest.json"}:
            raise ValueError("ZIP 목록 불일치")
        for name, digest in manifest["files"].items():
            # Reading to EOF validates CRC while SHA is computed in bounded memory.
            h = hashlib.sha256()
            with z.open(name) as stream:
                for block in iter(lambda: stream.read(1024 * 1024), b""):
                    h.update(block)
            if h.hexdigest() != digest:
                raise ValueError("ZIP 해시 불일치: " + name)
        contract = json.loads(z.read("contract.json"))
        if contract.get("format") != "v4-independent-1":
            raise ValueError("V4 ZIP이 아닙니다.")
    return contract


def pack(root, output):
    root, output = Path(root), Path(output)
    if root.resolve() in output.resolve().parents:
        raise ValueError("출력 ZIP은 작업폴더 밖에 있어야 합니다.")
    output.parent.mkdir(parents=True, exist_ok=True)
    files = {
        p.relative_to(root).as_posix(): sha_file(p)
        for p in sorted(root.rglob("*"))
        if p.is_file() and p.name != "v4_manifest.json"
    }
    write_json(root / "v4_manifest.json", {"files": files})
    fd, temp = tempfile.mkstemp(
        prefix=output.name + ".", suffix=".tmp", dir=output.parent
    )
    os.close(fd)
    try:
        with zipfile.ZipFile(temp, "w", zipfile.ZIP_DEFLATED) as z:
            for name in [*files, "v4_manifest.json"]:
                z.write(root / name, name)
        verify_zip(temp)
        os.replace(temp, output)
    finally:
        if Path(temp).exists():
            Path(temp).unlink()
    return output


def restore(bundle, root, expected=None):
    contract = verify_zip(bundle)
    if expected and any(contract.get(key) != value for key, value in expected.items()):
        raise ValueError("ZIP의 실행·실험 식별자가 다릅니다.")
    root = Path(root)
    if root.exists():
        raise ValueError("새 작업폴더가 필요합니다.")
    with zipfile.ZipFile(bundle) as z:
        z.extractall(root)
    return contract


def arrays_sha(arrays):
    import numpy as np

    h = hashlib.sha256()
    for name, value in sorted(arrays.items()):
        a = np.ascontiguousarray(value)
        h.update(name.encode())
        h.update(str(a.shape).encode())
        h.update(str(a.dtype).encode())
        h.update(a.tobytes())
    return h.hexdigest()


In [ ]:
%%writefile "{V4_MODULE_DIR}/verification.py"
"""Explicit checks for optional live integrations, never a repeated full audit."""

from .services import Service


def integration_checks(
    root, use_llm=False, dense=False, backend="numpy", questions=(), service=None
):
    questions = tuple(q.strip() for q in questions if q.strip())
    if not (use_llm or dense or questions):
        raise ValueError(
            "Qwen·E5 연결 또는 확인할 질문을 선택하세요. 전체 검사는 실행하지 않습니다."
        )
    service = service or Service(root)
    result = {
        "scope": "selected live integrations and explicit questions only",
        "training_executed": False,
        "equation_audit_executed": False,
        "all_model_audit_executed": False,
        "semantic_accuracy_verified": False,
    }
    if dense:
        matches = service.search("V4 RAD LHS Adam L-BFGS 비교", True, backend)
        result["retrieval"] = {
            "passed": bool(matches),
            "backend": "E5/" + backend + " + BM25 + RRF",
            "matches": matches,
        }
    else:
        result["retrieval"] = {"status": "not_requested"}
    if use_llm and not questions:
        questions = (
            "wave2d 순방향 Adam의 저장 결과를 비교해줘",
            "wave2d 역문제 L-BFGS의 저장 결과를 비교해줘",
            "V4의 RAD와 LHS 설정 자료를 찾아줘",
        )
    answers = [
        {"question": q, **service.ask(q, use_llm, dense, backend)} for q in questions
    ]
    calls = sum(r["llm_called"] for r in answers)
    result["questions"] = answers
    result["question_count"] = len(answers)
    result["actual_llm_calls"] = calls
    result["qwen"] = (
        {"actual_call_confirmed": bool(calls)}
        if use_llm
        else {"status": "not_requested"}
    )
    return result


## 설정 · 실행

처음에는 저장 결과 조회가 선택됩니다. 학습은 계획 확인과 실행 허용이 모두 필요합니다.

In [ ]:
from pathlib import Path
from v4.interface import panel

local_folder = Path("/content")
drive_folder = Path("/content/drive/MyDrive/pinnV4")
if (local_folder / "V4_Results.zip").is_file():
    result_folder = local_folder
else:
    result_folder = drive_folder
    if not Path("/content/drive/MyDrive").is_dir():
        try:
            from google.colab import drive
            drive.mount("/content/drive")
        except (ImportError, Exception) as error:
            print("Drive 연결을 완료하지 못했습니다. 하단의 로컬 ZIP 사용으로 전환할 수 있습니다.")

V4_PANEL = panel(folder=str(result_folder))
if (result_folder / "V4_Results.zip").is_file():
    V4_PANEL["execute"]()


## 학습 실행

학습할 때만 실행합니다. 위 패널에서 학습 실행 허용과 설정 확인·실행 준비를 완료한 뒤 이 셀의 ▶를 누르세요. Adam·L-BFGS·저장은 이 셀에서 실행되며 로그도 아래에 표시됩니다. 저장 결과 조회만 할 때는 실행하지 않습니다.

In [ ]:
# 학습 실행: 실행 준비를 마친 경우에만 학습합니다.
if "V4_PANEL" not in globals():
    raise RuntimeError("먼저 위의 설정·실행 패널 셀을 실행하세요.")
if V4_PANEL["state"].get("pending_training") is not None:
    V4_PANEL["run_training"]()
else:
    print("학습 준비 없음. 저장 결과 조회에는 이 셀 실행이 필요하지 않습니다.")
